# Quantity Cleaning

This notebook builds one final cleaned output view named `quantity_cleaned`.

The source parquet is never modified. The notebook reads the raw quantity fields, applies the cleaning rules, and produces:

- `quantity_cleaned` as the only final output view
- `quantity_cleaning_debug` as an optional internal review view
- `quantity_change_audit` as an optional internal change-audit view


## Notebook Flow

The notebook follows a simple layout:

1. Load the raw quantity fields without changing source data.
2. Build an internal reference snapshot only for audit checks.
3. Set up the current helper rules, aliases, and safe OCR cleanup.
4. Parse the free-text `quantity` field and compare it with the structured fields.
5. Produce one final cleaned view: `quantity_cleaned`.
6. Run optional debug and audit tables so we can understand unresolved, conflict, and rule-impact cases.


In [ ]:
# This cell imports the small Python helpers we need around DuckDB.
# DuckDB does the parsing and cleaning work. Python is only used to:
# - locate the parquet file
# - build SQL snippets from the approved alias lists
# - display query results in notebook-friendly tables

import json
import re
from pathlib import Path

import duckdb
from IPython.display import display


PARQUET_CANDIDATES = [
    Path("../data/raw/off-canada.parquet"),
    Path("data/raw/off-canada.parquet"),
    Path("../data/off-canada.parquet"),
    Path("data/off-canada.parquet"),
]

SOURCE_COLUMNS = ["code", "product_quantity_unit", "product_quantity", "quantity"]
VALUE_MATCH_TOLERANCE = 0.02
ARTICLE_TOKENS = {"a", "an", "un", "une"}

# Reviewed unit aliases only. These are deterministic and map to canonical base units.
MEASURE_ALIASES = [
    ("g", "g", "mass", 1.0),
    ("gram", "g", "mass", 1.0),
    ("grams", "g", "mass", 1.0),
    ("gramme", "g", "mass", 1.0),
    ("grammes", "g", "mass", 1.0),
    ("kg", "g", "mass", 1000.0),
    ("kilogram", "g", "mass", 1000.0),
    ("kilograms", "g", "mass", 1000.0),
    ("kilogramme", "g", "mass", 1000.0),
    ("kilogrammes", "g", "mass", 1000.0),
    ("oz", "g", "mass", 28.349523125),
    ("lb", "g", "mass", 453.59237),
    ("lbs", "g", "mass", 453.59237),
    ("ml", "ml", "volume", 1.0),
    ("millilitre", "ml", "volume", 1.0),
    ("millilitres", "ml", "volume", 1.0),
    ("milliliter", "ml", "volume", 1.0),
    ("milliliters", "ml", "volume", 1.0),
    ("l", "ml", "volume", 1000.0),
    ("litre", "ml", "volume", 1000.0),
    ("litres", "ml", "volume", 1000.0),
    ("liter", "ml", "volume", 1000.0),
    ("liters", "ml", "volume", 1000.0),
    ("lit", "ml", "volume", 1000.0),
    ("cl", "ml", "volume", 10.0),
    ("dl", "ml", "volume", 100.0),
    ("fl oz", "ml", "volume", 29.5735295625),
    ("kj", "kj", "energy", 1.0),
    ("kcal", "kcal", "energy", 1.0),
    ("cal", "cal", "energy", 1.0),
]

# Count descriptors describe discrete items whose quantity is meaningful by count.
COUNT_DESCRIPTOR_ALIASES = [
    ("pc", "piece", "count"),
    ("pcs", "piece", "count"),
    ("piece", "piece", "count"),
    ("pieces", "piece", "count"),
    ("ea", "piece", "count"),
    ("each", "piece", "count"),
    ("tablet", "tablet", "count"),
    ("tablets", "tablet", "count"),
    ("tablilla", "tablet", "count"),
    ("tablillas", "tablet", "count"),
    ("capsule", "capsule", "count"),
    ("capsules", "capsule", "count"),
    ("cap", "capsule", "count"),
    ("caps", "capsule", "count"),
    ("sachet", "sachet", "count"),
    ("sachets", "sachet", "count"),
    ("morceau", "piece", "count"),
    ("morceaux", "piece", "count"),
    ("bottle", "bottle", "count"),
    ("bottles", "bottle", "count"),
    ("bouteille", "bottle", "count"),
    ("bouteilles", "bottle", "count"),
    ("can", "can", "count"),
    ("cans", "can", "count"),
    ("canette", "can", "count"),
    ("canettes", "can", "count"),
]

# Packaging descriptors are meaningful, but by themselves they do not give a comparable size.
PACKAGING_DESCRIPTOR_ALIASES = [
    ("box", "box", "packaging_only"),
    ("boxes", "box", "packaging_only"),
    ("boite", "box", "packaging_only"),
    ("boites", "box", "packaging_only"),
    ("pack", "pack", "packaging_only"),
    ("packs", "pack", "packaging_only"),
    ("paquet", "pack", "packaging_only"),
    ("paquets", "pack", "packaging_only"),
    ("carton", "carton", "packaging_only"),
    ("cartons", "carton", "packaging_only"),
    ("case", "case", "packaging_only"),
    ("cases", "case", "packaging_only"),
    ("caisse", "case", "packaging_only"),
    ("caisses", "case", "packaging_only"),
    ("tray", "tray", "packaging_only"),
    ("trays", "tray", "packaging_only"),
]

# Household and serving units are intentionally kept out of package-size normalization.
HOUSEHOLD_ALIASES = [
    ("tsp", "tsp"),
    ("tbsp", "tbsp"),
    ("cup", "cup"),
    ("cups", "cup"),
    ("tasse", "cup"),
    ("tasses", "cup"),
    ("portion", "portion"),
    ("portions", "portion"),
    ("serving", "serving"),
    ("servings", "serving"),
]

PLACEHOLDER_TERMS = [
    "unknown",
    "unknown quantity",
    "n/a",
    "none",
    "not labeled!",
    "good",
    "bonne",
    "bn batouta",
]


In [ ]:
# This cell defines the small helper functions used by the notebook.
# They keep the SQL readable while making the logic reproducible.

def resolve_parquet_path(candidates: list[Path]) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Could not find the OFF Canada parquet. Checked:\n{checked}")


def sql_quote(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


def build_token_pattern(tokens: list[str]) -> str:
    escaped_tokens = []
    for token in sorted(set(tokens), key=len, reverse=True):
        escaped = re.escape(token).replace(r"\ ", r"\s*")
        escaped_tokens.append(escaped)
    return "(?:" + "|".join(escaped_tokens) + ")"


def normalize_sql(expr: str) -> str:
    normalized = f"lower(trim(coalesce({expr}, '')))"
    replacements = [
        ("’", "'"),
        ("`", "'"),
        ("´", "'"),
        ("×", "x"),
        ("é", "e"),
        ("è", "e"),
        ("ê", "e"),
        ("ë", "e"),
        ("à", "a"),
        ("â", "a"),
        ("ä", "a"),
        ("î", "i"),
        ("ï", "i"),
        ("ô", "o"),
        ("ö", "o"),
        ("ù", "u"),
        ("û", "u"),
        ("ü", "u"),
        ("ç", "c"),
    ]
    for raw_text, canonical_text in replacements:
        normalized = f"replace({normalized}, {sql_quote(raw_text)}, {sql_quote(canonical_text)})"
    normalized = f"regexp_replace({normalized}, '\\s+', ' ', 'g')"
    return normalized


def values_sql(rows: list[tuple]) -> str:
    return ",\n".join("(" + ", ".join(sql_quote(str(value)) if isinstance(value, str) else str(value) for value in row) + ")" for row in rows)


def show_query(title: str, query: str, limit: int | None = None) -> None:
    final_query = query if limit is None else f"{query}\nLIMIT {limit}"
    print(title)
    display(con.sql(final_query).df())


PARQUET_PATH = resolve_parquet_path(PARQUET_CANDIDATES)
con = duckdb.connect()

descriptor_rows = COUNT_DESCRIPTOR_ALIASES + PACKAGING_DESCRIPTOR_ALIASES

con.execute(
    f'''
    CREATE OR REPLACE TEMP VIEW measure_aliases AS
    SELECT *
    FROM (VALUES
    {values_sql(MEASURE_ALIASES)}
    ) AS alias_rows(token, base_unit, quantity_category, factor);
    '''
)

con.execute(
    f'''
    CREATE OR REPLACE TEMP VIEW descriptor_aliases AS
    SELECT *
    FROM (VALUES
    {values_sql(descriptor_rows)}
    ) AS alias_rows(token, item_descriptor, quantity_category);
    '''
)

con.execute(
    f'''
    CREATE OR REPLACE TEMP VIEW household_aliases AS
    SELECT *
    FROM (VALUES
    {values_sql(HOUSEHOLD_ALIASES)}
    ) AS alias_rows(token, item_descriptor);
    '''
)

NUMBER_PATTERN = r"[0-9]+(?:[.,][0-9]+)?"
FRACTION_OR_NUMBER_PATTERN = rf"(?:[0-9]+/[0-9]+|{NUMBER_PATTERN})"
MEASURE_PATTERN = build_token_pattern([row[0] for row in MEASURE_ALIASES])
COUNT_DESCRIPTOR_PATTERN = build_token_pattern([row[0] for row in COUNT_DESCRIPTOR_ALIASES])
PACKAGING_DESCRIPTOR_PATTERN = build_token_pattern([row[0] for row in PACKAGING_DESCRIPTOR_ALIASES])
DESCRIPTOR_PATTERN = build_token_pattern([row[0] for row in descriptor_rows])
HOUSEHOLD_PATTERN = build_token_pattern([row[0] for row in HOUSEHOLD_ALIASES])
MEASURE_ANY_REGEX = rf"\b{MEASURE_PATTERN}\b"
HOUSEHOLD_ANY_REGEX = rf"\b{HOUSEHOLD_PATTERN}\b"
PLACEHOLDER_PATTERN = build_token_pattern(PLACEHOLDER_TERMS)
PLACEHOLDER_REGEX = rf"^(?:{PLACEHOLDER_PATTERN}|\?+)$"

SIMPLE_MEASURE_REGEX = rf"^\s*({NUMBER_PATTERN})\s*({MEASURE_PATTERN})\s*[.]?\s*$"
MULTIPACK_MEASURE_REGEX = rf"({NUMBER_PATTERN})\s*[*x]\s*({NUMBER_PATTERN})\s*({MEASURE_PATTERN})\b"
DESCRIPTOR_MULTIPACK_REGEX = rf"^\s*({NUMBER_PATTERN}|un|une|a|an)\s*({DESCRIPTOR_PATTERN})\b\s*(?:de|of|[*x])\s*({NUMBER_PATTERN})\s*({MEASURE_PATTERN})\b(?:\s*(?:chacun|chacune|each))?\s*[.]?\s*$"
DESCRIPTOR_WITH_MEASURE_REGEX = rf"^\s*({NUMBER_PATTERN}|un|une|a|an)?\s*({DESCRIPTOR_PATTERN})\b.*?({NUMBER_PATTERN})\s*({MEASURE_PATTERN})\b"
DESCRIPTOR_ONLY_REGEX = rf"^\s*({NUMBER_PATTERN}|un|une|a|an)?\s*({DESCRIPTOR_PATTERN})\b.*$"
PER_PACKAGING_REGEX = rf"^\s*({NUMBER_PATTERN})\s+par\s+({PACKAGING_DESCRIPTOR_PATTERN})\b"
HOUSEHOLD_REGEX = rf"^\s*({FRACTION_OR_NUMBER_PATTERN})\s*({HOUSEHOLD_PATTERN})\b"
NUMBER_ONLY_REGEX = rf"^\s*({NUMBER_PATTERN})\s*$"

print(f"Using parquet: {PARQUET_PATH}")


In [ ]:
# This cell loads only the raw fields needed for the quantity cleaner.
# A notebook-local row id is added so later debugging stays easy.

quantity_raw_sql = f'''
CREATE OR REPLACE TEMP VIEW quantity_raw AS
SELECT
    row_number() OVER () AS row_id,
    {", ".join(SOURCE_COLUMNS)}
FROM read_parquet({sql_quote(PARQUET_PATH.as_posix())});
'''

con.execute(quantity_raw_sql)

show_query("Raw quantity rows", "SELECT COUNT(*) AS row_count FROM quantity_raw")
show_query("Sample raw rows", "SELECT * FROM quantity_raw ORDER BY row_id", limit=10)


In [ ]:
# This cell rebuilds the earlier reference text-parser for internal audit use only.
# It is not the final output of the notebook.

# This cell creates the feature view that parses the raw fields into candidate signals.
# The view does not make the final decision yet. It only extracts everything needed for:
# - structured product field parsing
# - free-text quantity parsing
# - later cross-field comparison

quantity_features_sql = f'''
CREATE OR REPLACE TEMP VIEW quantity_cleaning_features_reference AS
WITH base AS (
    SELECT
        row_id,
        code,
        product_quantity_unit,
        {normalize_sql("product_quantity_unit")} AS product_quantity_unit_normalized,
        product_quantity,
        TRY_CAST(replace(NULLIF(trim(product_quantity), ''), ',', '.') AS DOUBLE) AS product_quantity_numeric,
        quantity,
        {normalize_sql("quantity")} AS quantity_normalized
    FROM quantity_raw
),
extracted AS (
    SELECT
        b.*,
        NULLIF(regexp_extract(quantity_normalized, '{MULTIPACK_MEASURE_REGEX}', 1), '') AS multipack_count_text,
        NULLIF(regexp_extract(quantity_normalized, '{MULTIPACK_MEASURE_REGEX}', 2), '') AS multipack_inner_value_text,
        NULLIF(regexp_extract(quantity_normalized, '{MULTIPACK_MEASURE_REGEX}', 3), '') AS multipack_unit_token,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_MULTIPACK_REGEX}', 1), '') AS descriptor_multipack_count_text,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_MULTIPACK_REGEX}', 2), '') AS descriptor_multipack_token,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_MULTIPACK_REGEX}', 3), '') AS descriptor_multipack_inner_value_text,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_MULTIPACK_REGEX}', 4), '') AS descriptor_multipack_unit_token,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_WITH_MEASURE_REGEX}', 1), '') AS descriptor_measure_leading_token,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_WITH_MEASURE_REGEX}', 2), '') AS descriptor_measure_token,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_WITH_MEASURE_REGEX}', 3), '') AS descriptor_measure_value_text,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_WITH_MEASURE_REGEX}', 4), '') AS descriptor_measure_unit_token,
        NULLIF(regexp_extract(quantity_normalized, '{SIMPLE_MEASURE_REGEX}', 1), '') AS simple_value_text,
        NULLIF(regexp_extract(quantity_normalized, '{SIMPLE_MEASURE_REGEX}', 2), '') AS simple_unit_token,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_ONLY_REGEX}', 1), '') AS descriptor_only_leading_token,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_ONLY_REGEX}', 2), '') AS descriptor_only_token,
        NULLIF(regexp_extract(quantity_normalized, '{PER_PACKAGING_REGEX}', 1), '') AS per_pack_count_text,
        NULLIF(regexp_extract(quantity_normalized, '{PER_PACKAGING_REGEX}', 2), '') AS per_pack_token,
        NULLIF(regexp_extract(quantity_normalized, '{HOUSEHOLD_REGEX}', 1), '') AS household_value_text,
        NULLIF(regexp_extract(quantity_normalized, '{HOUSEHOLD_REGEX}', 2), '') AS household_unit_token,
        regexp_matches(quantity_normalized, '{PLACEHOLDER_REGEX}') AS quantity_is_placeholder,
        regexp_matches(quantity_normalized, '{NUMBER_ONLY_REGEX}') AS quantity_is_number_only,
        regexp_matches(quantity_normalized, '{HOUSEHOLD_ANY_REGEX}') AS quantity_has_household_token,
        regexp_matches(quantity_normalized, '{MEASURE_ANY_REGEX}') AS quantity_has_measure_token,
        regexp_matches(quantity_normalized, '\\d') AS quantity_has_digits
    FROM base b
),
joined AS (
    SELECT
        e.*,
        pm.base_unit AS product_base_unit,
        pm.quantity_category AS product_category,
        pm.factor AS product_factor,
        mm.base_unit AS multipack_base_unit,
        mm.quantity_category AS multipack_category,
        mm.factor AS multipack_factor,
        dpm.base_unit AS descriptor_multipack_base_unit,
        dpm.quantity_category AS descriptor_multipack_category,
        dpm.factor AS descriptor_multipack_factor,
        dm.base_unit AS descriptor_measure_base_unit,
        dm.quantity_category AS descriptor_measure_category,
        dm.factor AS descriptor_measure_factor,
        sm.base_unit AS simple_base_unit,
        sm.quantity_category AS simple_category,
        sm.factor AS simple_factor,
        dpd.item_descriptor AS descriptor_multipack_item_descriptor,
        de.item_descriptor AS descriptor_measure_item_descriptor,
        de.quantity_category AS descriptor_measure_descriptor_category,
        d0.item_descriptor AS descriptor_only_item_descriptor,
        d0.quantity_category AS descriptor_only_category,
        pp.item_descriptor AS per_pack_item_descriptor,
        hh.item_descriptor AS household_descriptor
    FROM extracted e
    LEFT JOIN measure_aliases pm ON e.product_quantity_unit_normalized = pm.token
    LEFT JOIN measure_aliases mm ON e.multipack_unit_token = mm.token
    LEFT JOIN measure_aliases dpm ON e.descriptor_multipack_unit_token = dpm.token
    LEFT JOIN measure_aliases dm ON e.descriptor_measure_unit_token = dm.token
    LEFT JOIN measure_aliases sm ON e.simple_unit_token = sm.token
    LEFT JOIN descriptor_aliases dpd ON e.descriptor_multipack_token = dpd.token
    LEFT JOIN descriptor_aliases de ON e.descriptor_measure_token = de.token
    LEFT JOIN descriptor_aliases d0 ON e.descriptor_only_token = d0.token
    LEFT JOIN descriptor_aliases pp ON e.per_pack_token = pp.token
    LEFT JOIN household_aliases hh ON e.household_unit_token = hh.token
),
typed AS (
    SELECT
        *,
        CASE
            WHEN lower(coalesce(descriptor_multipack_count_text, '')) IN ('un', 'une', 'a', 'an') THEN 1.0
            ELSE TRY_CAST(replace(descriptor_multipack_count_text, ',', '.') AS DOUBLE)
        END AS descriptor_multipack_count_numeric,
        CASE
            WHEN lower(coalesce(descriptor_measure_leading_token, '')) IN ('un', 'une', 'a', 'an') THEN 1.0
            ELSE TRY_CAST(replace(descriptor_measure_leading_token, ',', '.') AS DOUBLE)
        END AS descriptor_measure_count_numeric,
        CASE
            WHEN lower(coalesce(descriptor_only_leading_token, '')) IN ('un', 'une', 'a', 'an') THEN 1.0
            ELSE TRY_CAST(replace(descriptor_only_leading_token, ',', '.') AS DOUBLE)
        END AS descriptor_only_count_numeric,
        TRY_CAST(replace(descriptor_multipack_inner_value_text, ',', '.') AS DOUBLE) AS descriptor_multipack_inner_value_numeric,
        TRY_CAST(replace(multipack_count_text, ',', '.') AS DOUBLE) AS multipack_count_numeric,
        TRY_CAST(replace(multipack_inner_value_text, ',', '.') AS DOUBLE) AS multipack_inner_value_numeric,
        TRY_CAST(replace(descriptor_measure_value_text, ',', '.') AS DOUBLE) AS descriptor_measure_value_numeric,
        TRY_CAST(replace(simple_value_text, ',', '.') AS DOUBLE) AS simple_value_numeric,
        TRY_CAST(replace(per_pack_count_text, ',', '.') AS DOUBLE) AS per_pack_count_numeric,
        CASE
            WHEN household_value_text LIKE '%/%' THEN
                TRY_CAST(split_part(household_value_text, '/', 1) AS DOUBLE)
                / NULLIF(TRY_CAST(split_part(household_value_text, '/', 2) AS DOUBLE), 0)
            ELSE TRY_CAST(replace(household_value_text, ',', '.') AS DOUBLE)
        END AS household_value_numeric,
        CASE
            WHEN product_quantity_numeric IS NOT NULL AND product_factor IS NOT NULL THEN product_quantity_numeric * product_factor
        END AS product_normalized_value,
        CASE
            WHEN multipack_inner_value_numeric IS NOT NULL
                 AND multipack_factor IS NOT NULL
            THEN multipack_inner_value_numeric * multipack_factor
        END AS multipack_inner_normalized_value,
        CASE
            WHEN multipack_count_numeric IS NOT NULL
                 AND multipack_inner_value_numeric IS NOT NULL
                 AND multipack_factor IS NOT NULL
            THEN multipack_count_numeric * multipack_inner_value_numeric * multipack_factor
        END AS multipack_normalized_value,
        CASE
            WHEN descriptor_multipack_count_numeric IS NOT NULL
                 AND descriptor_multipack_count_numeric > 1
                 AND descriptor_multipack_inner_value_numeric IS NOT NULL
                 AND descriptor_multipack_factor IS NOT NULL
            THEN descriptor_multipack_inner_value_numeric * descriptor_multipack_factor
        END AS descriptor_multipack_inner_normalized_value,
        CASE
            WHEN descriptor_multipack_count_numeric IS NOT NULL
                 AND descriptor_multipack_count_numeric > 1
                 AND descriptor_multipack_inner_value_numeric IS NOT NULL
                 AND descriptor_multipack_factor IS NOT NULL
            THEN descriptor_multipack_count_numeric * descriptor_multipack_inner_value_numeric * descriptor_multipack_factor
        END AS descriptor_multipack_total_normalized_value,
        CASE
            WHEN descriptor_measure_value_numeric IS NOT NULL
                 AND descriptor_measure_factor IS NOT NULL
            THEN descriptor_measure_value_numeric * descriptor_measure_factor
        END AS descriptor_measure_normalized_value,
        CASE
            WHEN simple_value_numeric IS NOT NULL AND simple_factor IS NOT NULL
            THEN simple_value_numeric * simple_factor
        END AS simple_normalized_value
    FROM joined
)
SELECT
    *,
    CASE
        WHEN quantity IS NULL THEN NULL
        WHEN quantity_normalized = '' THEN 'unresolved'
        WHEN quantity_is_placeholder THEN 'unresolved'
        WHEN quantity_has_household_token THEN 'unresolved'
        WHEN multipack_normalized_value IS NOT NULL AND multipack_category IN ('mass', 'volume') THEN 'resolved'
        WHEN descriptor_multipack_total_normalized_value IS NOT NULL AND descriptor_multipack_category IN ('mass', 'volume') THEN 'resolved'
        WHEN descriptor_measure_normalized_value IS NOT NULL AND descriptor_measure_category = 'energy' THEN 'partial'
        WHEN descriptor_measure_normalized_value IS NOT NULL AND descriptor_measure_category IN ('mass', 'volume') THEN 'resolved'
        WHEN simple_normalized_value IS NOT NULL AND simple_category = 'energy' THEN 'partial'
        WHEN simple_normalized_value IS NOT NULL AND simple_category IN ('mass', 'volume') THEN 'resolved'
        WHEN descriptor_only_item_descriptor IS NOT NULL AND descriptor_only_category = 'count' THEN 'resolved'
        WHEN descriptor_only_item_descriptor IS NOT NULL AND descriptor_only_category = 'packaging_only' THEN 'partial'
        WHEN per_pack_item_descriptor IS NOT NULL THEN 'partial'
        WHEN quantity_is_number_only THEN 'unresolved'
        WHEN NOT quantity_has_digits THEN 'unresolved'
        ELSE 'unresolved'
    END AS text_status,
    CASE
        WHEN quantity IS NULL THEN NULL
        WHEN quantity_normalized = '' THEN NULL
        WHEN quantity_is_placeholder THEN 'placeholder_or_unknown'
        WHEN quantity_has_household_token THEN 'household_unit'
        WHEN multipack_normalized_value IS NOT NULL AND multipack_category IN ('mass', 'volume') THEN 'multipack_measure'
        WHEN descriptor_multipack_total_normalized_value IS NOT NULL AND descriptor_multipack_category IN ('mass', 'volume') THEN 'multipack_measure'
        WHEN descriptor_measure_normalized_value IS NOT NULL THEN descriptor_measure_category
        WHEN simple_normalized_value IS NOT NULL THEN simple_category
        WHEN descriptor_only_item_descriptor IS NOT NULL THEN descriptor_only_category
        WHEN per_pack_item_descriptor IS NOT NULL THEN 'packaging_only'
        WHEN quantity_has_measure_token THEN 'mixed_measure'
        WHEN NOT quantity_has_digits THEN 'noise_or_non_quantity'
        ELSE NULL
    END AS text_category,
    CASE
        WHEN multipack_normalized_value IS NOT NULL AND multipack_category IN ('mass', 'volume') THEN multipack_normalized_value
        WHEN descriptor_multipack_total_normalized_value IS NOT NULL AND descriptor_multipack_category IN ('mass', 'volume') THEN descriptor_multipack_total_normalized_value
        WHEN descriptor_measure_normalized_value IS NOT NULL THEN descriptor_measure_normalized_value
        WHEN simple_normalized_value IS NOT NULL THEN simple_normalized_value
        WHEN descriptor_only_item_descriptor IS NOT NULL AND descriptor_only_category = 'count' THEN coalesce(descriptor_only_count_numeric, 1.0)
        WHEN descriptor_only_item_descriptor IS NOT NULL AND descriptor_only_category = 'packaging_only' THEN coalesce(descriptor_only_count_numeric, 1.0)
        WHEN per_pack_item_descriptor IS NOT NULL THEN per_pack_count_numeric
        ELSE NULL
    END AS text_normalized_value,
    CASE
        WHEN multipack_normalized_value IS NOT NULL AND multipack_category IN ('mass', 'volume') THEN multipack_base_unit
        WHEN descriptor_multipack_total_normalized_value IS NOT NULL AND descriptor_multipack_category IN ('mass', 'volume') THEN descriptor_multipack_base_unit
        WHEN descriptor_measure_normalized_value IS NOT NULL THEN descriptor_measure_base_unit
        WHEN simple_normalized_value IS NOT NULL THEN simple_base_unit
        WHEN descriptor_only_item_descriptor IS NOT NULL THEN 'count'
        WHEN per_pack_item_descriptor IS NOT NULL THEN 'count'
        ELSE NULL
    END AS text_normalized_unit,
    CASE
        WHEN multipack_normalized_value IS NOT NULL AND multipack_category IN ('mass', 'volume') THEN multipack_inner_normalized_value
        WHEN descriptor_multipack_total_normalized_value IS NOT NULL AND descriptor_multipack_category IN ('mass', 'volume') THEN descriptor_multipack_inner_normalized_value
        ELSE NULL
    END AS text_inner_normalized_value,
    CASE
        WHEN multipack_normalized_value IS NOT NULL AND multipack_category IN ('mass', 'volume') THEN multipack_count_numeric
        WHEN descriptor_multipack_total_normalized_value IS NOT NULL AND descriptor_multipack_category IN ('mass', 'volume') THEN descriptor_multipack_count_numeric
        WHEN descriptor_measure_normalized_value IS NOT NULL AND descriptor_measure_category IN ('mass', 'volume', 'energy') THEN coalesce(descriptor_measure_count_numeric, 1.0)
        WHEN descriptor_only_item_descriptor IS NOT NULL AND descriptor_only_category = 'packaging_only' THEN coalesce(descriptor_only_count_numeric, 1.0)
        WHEN per_pack_item_descriptor IS NOT NULL THEN per_pack_count_numeric
        WHEN simple_normalized_value IS NOT NULL THEN 1.0
        ELSE NULL
    END AS text_pack_count,
    CASE
        WHEN descriptor_multipack_item_descriptor IS NOT NULL THEN descriptor_multipack_item_descriptor
        WHEN descriptor_measure_item_descriptor IS NOT NULL THEN descriptor_measure_item_descriptor
        WHEN descriptor_only_item_descriptor IS NOT NULL THEN descriptor_only_item_descriptor
        WHEN per_pack_item_descriptor IS NOT NULL THEN per_pack_item_descriptor
        ELSE NULL
    END AS text_item_descriptor,
    CASE
        WHEN quantity IS NULL THEN 'quantity missing'
        WHEN quantity_normalized = '' THEN 'blank quantity string'
        WHEN quantity_is_placeholder THEN 'placeholder or unknown text'
        WHEN quantity_has_household_token THEN 'household unit not used for consolidation'
        WHEN multipack_normalized_value IS NOT NULL AND multipack_category IN ('mass', 'volume') THEN 'derived total from quantity multipack'
        WHEN descriptor_multipack_total_normalized_value IS NOT NULL AND descriptor_multipack_category IN ('mass', 'volume') THEN 'derived total from quantity descriptor multipack'
        WHEN descriptor_measure_normalized_value IS NOT NULL AND descriptor_measure_category = 'energy' THEN 'energy text captured but not used for consolidation'
        WHEN descriptor_measure_normalized_value IS NOT NULL AND descriptor_measure_category IN ('mass', 'volume') THEN 'descriptor plus measure from quantity'
        WHEN simple_normalized_value IS NOT NULL AND simple_category = 'energy' THEN 'energy text captured but not used for consolidation'
        WHEN simple_normalized_value IS NOT NULL AND simple_category IN ('mass', 'volume') THEN 'simple measure from quantity'
        WHEN descriptor_only_item_descriptor IS NOT NULL AND descriptor_only_category = 'count' THEN 'count descriptor from quantity'
        WHEN descriptor_only_item_descriptor IS NOT NULL AND descriptor_only_category = 'packaging_only' THEN 'packaging only, no comparable size'
        WHEN per_pack_item_descriptor IS NOT NULL THEN 'per packaging phrase, no comparable size'
        WHEN quantity_is_number_only THEN 'number only, unit missing'
        WHEN NOT quantity_has_digits THEN 'non-quantity text in quantity field'
        WHEN quantity_has_measure_token THEN 'mixed or unsupported measure expression'
        ELSE 'unparsed quantity text'
    END AS text_note
FROM typed;
'''

con.execute(quantity_features_sql)

print("Reference text-parser view ready for internal change audit.")


In [ ]:
# This cell rebuilds the earlier reference cleaned view for internal audit use only.
# It is not the final output of the notebook.

# This cell applies the final cross-field decision rules and produces the reference cleaned view.
# The logic is intentionally conservative:
# - use structured product fields when they are clear
# - use quantity text when it cleanly fills missing structure
# - keep packaging-only and household cases visible
# - mark real disagreements as conflicts instead of guessing

quantity_cleaned_sql = f'''
CREATE OR REPLACE TEMP VIEW quantity_cleaned_reference AS
WITH staged AS (
    SELECT
        *,
        CASE
            WHEN product_normalized_value IS NOT NULL AND product_category IN ('mass', 'volume') THEN 'resolved'
            WHEN product_normalized_value IS NOT NULL AND product_category = 'energy' THEN 'partial'
            ELSE NULL
        END AS product_status,
        CASE
            WHEN product_category IN ('mass', 'volume', 'energy')
                 AND text_normalized_value IS NOT NULL
                 AND text_normalized_unit IS NOT NULL
            THEN product_base_unit = text_normalized_unit
            ELSE FALSE
        END AS comparable_unit_match,
        CASE
            WHEN product_category IN ('mass', 'volume', 'energy')
                 AND text_normalized_value IS NOT NULL
                 AND text_normalized_unit IS NOT NULL
                 AND product_base_unit = text_normalized_unit
            THEN abs(product_normalized_value - text_normalized_value)
                 <= greatest(0.01, {VALUE_MATCH_TOLERANCE} * greatest(abs(product_normalized_value), abs(text_normalized_value)))
            ELSE FALSE
        END AS comparable_value_match,
        CASE
            WHEN product_category IN ('mass', 'volume')
                 AND text_category = 'multipack_measure'
                 AND text_inner_normalized_value IS NOT NULL
                 AND product_base_unit = text_normalized_unit
            THEN abs(product_normalized_value - text_inner_normalized_value)
                 <= greatest(0.01, {VALUE_MATCH_TOLERANCE} * greatest(abs(product_normalized_value), abs(text_inner_normalized_value)))
            ELSE FALSE
        END AS multipack_inner_value_match,
        CASE
            WHEN product_quantity_numeric IS NOT NULL
                 AND text_category = 'multipack_measure'
                 AND text_normalized_value IS NOT NULL
            THEN (
                abs(product_quantity_numeric - text_normalized_value)
                <= greatest(0.01, {VALUE_MATCH_TOLERANCE} * greatest(abs(product_quantity_numeric), abs(text_normalized_value)))
            ) OR (
                text_inner_normalized_value IS NOT NULL
                AND abs(product_quantity_numeric - text_inner_normalized_value)
                    <= greatest(0.01, {VALUE_MATCH_TOLERANCE} * greatest(abs(product_quantity_numeric), abs(text_inner_normalized_value)))
            )
            WHEN product_quantity_numeric IS NOT NULL
                 AND text_normalized_value IS NOT NULL
                 AND text_normalized_unit = 'count'
            THEN product_quantity_numeric = text_normalized_value
            WHEN product_quantity_numeric IS NOT NULL
                 AND text_normalized_value IS NOT NULL
            THEN abs(product_quantity_numeric - text_normalized_value)
                 <= greatest(0.01, {VALUE_MATCH_TOLERANCE} * greatest(abs(product_quantity_numeric), abs(text_normalized_value)))
            ELSE FALSE
        END AS raw_numeric_matches_text
    FROM quantity_cleaning_features_reference
)
SELECT
    row_id,
    code,
    product_quantity_unit,
    product_quantity,
    quantity,
    CASE
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('mass', 'volume', 'multipack_measure', 'energy')
             AND NOT comparable_value_match
             AND NOT multipack_inner_value_match
        THEN 'conflict'
        WHEN product_status = 'partial'
             AND text_status = 'partial'
             AND text_category = 'energy'
             AND comparable_value_match
        THEN 'partial'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND multipack_inner_value_match
        THEN 'resolved'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND comparable_value_match
        THEN 'resolved'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category IN ('mass', 'volume')
             AND comparable_value_match
        THEN 'resolved'
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('count', 'packaging_only')
        THEN 'resolved'
        WHEN product_status = 'resolved'
        THEN 'resolved'
        WHEN product_quantity_numeric = 0
             AND text_status IN ('resolved', 'partial')
             AND text_normalized_value IS NOT NULL
             AND coalesce(text_normalized_value, 0) > 0
        THEN text_status
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND raw_numeric_matches_text
        THEN text_status
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND NOT raw_numeric_matches_text
        THEN 'conflict'
        WHEN text_status IS NOT NULL
        THEN text_status
        WHEN product_quantity_numeric IS NOT NULL AND product_base_unit IS NULL
        THEN 'unresolved'
        ELSE 'unresolved'
    END AS quantity_status,
    CASE
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('mass', 'volume', 'multipack_measure', 'energy')
             AND NOT comparable_value_match
             AND NOT multipack_inner_value_match
        THEN product_category
        WHEN product_status = 'partial'
             AND text_status = 'partial'
             AND text_category = 'energy'
             AND comparable_value_match
        THEN 'energy'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND multipack_inner_value_match
        THEN 'multipack_measure'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND comparable_value_match
        THEN 'multipack_measure'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category IN ('mass', 'volume')
             AND comparable_value_match
        THEN product_category
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('count', 'packaging_only')
        THEN product_category
        WHEN product_status = 'resolved'
        THEN product_category
        WHEN product_quantity_numeric = 0
             AND text_status IN ('resolved', 'partial')
             AND text_normalized_value IS NOT NULL
             AND coalesce(text_normalized_value, 0) > 0
        THEN text_category
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND raw_numeric_matches_text
        THEN text_category
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND NOT raw_numeric_matches_text
        THEN text_category
        WHEN text_status IS NOT NULL
        THEN text_category
        ELSE NULL
    END AS quantity_category,
    CASE
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('mass', 'volume', 'multipack_measure', 'energy')
             AND NOT comparable_value_match
             AND NOT multipack_inner_value_match
        THEN NULL
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND multipack_inner_value_match
        THEN text_normalized_value
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND comparable_value_match
        THEN text_normalized_value
        WHEN product_status = 'resolved'
        THEN product_normalized_value
        WHEN product_status = 'partial'
             AND text_status = 'partial'
             AND text_category = 'energy'
             AND comparable_value_match
        THEN product_normalized_value
        WHEN product_quantity_numeric = 0
             AND text_status IN ('resolved', 'partial')
             AND text_normalized_value IS NOT NULL
             AND coalesce(text_normalized_value, 0) > 0
        THEN text_normalized_value
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND raw_numeric_matches_text
        THEN text_normalized_value
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND NOT raw_numeric_matches_text
        THEN NULL
        WHEN text_status IN ('resolved', 'partial')
        THEN text_normalized_value
        ELSE NULL
    END AS normalized_value,
    CASE
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('mass', 'volume', 'multipack_measure', 'energy')
             AND NOT comparable_value_match
             AND NOT multipack_inner_value_match
        THEN NULL
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND multipack_inner_value_match
        THEN text_normalized_unit
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND comparable_value_match
        THEN text_normalized_unit
        WHEN product_status = 'resolved'
        THEN product_base_unit
        WHEN product_status = 'partial'
             AND text_status = 'partial'
             AND text_category = 'energy'
             AND comparable_value_match
        THEN product_base_unit
        WHEN product_quantity_numeric = 0
             AND text_status IN ('resolved', 'partial')
             AND text_normalized_value IS NOT NULL
             AND coalesce(text_normalized_value, 0) > 0
        THEN text_normalized_unit
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND raw_numeric_matches_text
        THEN text_normalized_unit
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND NOT raw_numeric_matches_text
        THEN NULL
        WHEN text_status IN ('resolved', 'partial')
        THEN text_normalized_unit
        ELSE NULL
    END AS normalized_unit,
    CASE
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND multipack_inner_value_match
        THEN text_pack_count
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND comparable_value_match
        THEN text_pack_count
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('count', 'packaging_only')
        THEN coalesce(text_pack_count, 1.0)
        WHEN product_status = 'resolved'
        THEN 1.0
        WHEN product_status = 'partial'
             AND text_status = 'partial'
             AND text_category = 'energy'
             AND comparable_value_match
        THEN coalesce(text_pack_count, 1.0)
        WHEN product_quantity_numeric = 0
             AND text_status IN ('resolved', 'partial')
        THEN coalesce(text_pack_count, CASE WHEN text_category IN ('count', 'packaging_only') THEN text_normalized_value ELSE 1.0 END)
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND raw_numeric_matches_text
        THEN coalesce(text_pack_count, CASE WHEN text_category IN ('count', 'packaging_only') THEN text_normalized_value ELSE 1.0 END)
        WHEN text_status IN ('resolved', 'partial')
        THEN coalesce(text_pack_count, CASE WHEN text_category IN ('count', 'packaging_only') THEN text_normalized_value ELSE 1.0 END)
        ELSE NULL
    END AS pack_count,
    CASE
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('count', 'packaging_only', 'multipack_measure', 'mass', 'volume', 'mixed_measure')
        THEN text_item_descriptor
        WHEN product_status = 'resolved'
        THEN NULL
        WHEN product_status = 'partial'
             AND text_status = 'partial'
             AND text_category = 'energy'
             AND comparable_value_match
        THEN text_item_descriptor
        WHEN product_quantity_numeric = 0
             AND text_status IN ('resolved', 'partial')
        THEN text_item_descriptor
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND raw_numeric_matches_text
        THEN text_item_descriptor
        WHEN text_status IN ('resolved', 'partial')
        THEN text_item_descriptor
        ELSE NULL
    END AS item_descriptor,
    CASE
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('mass', 'volume', 'multipack_measure', 'energy')
             AND NOT comparable_value_match
             AND NOT multipack_inner_value_match
        THEN TRUE
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND NOT raw_numeric_matches_text
        THEN TRUE
        ELSE FALSE
    END AS quantity_conflict_flag,
    CASE
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND multipack_inner_value_match
        THEN 'product fields match inner quantity; total derived from quantity multipack'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND comparable_value_match
        THEN 'structured total matches quantity multipack'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category IN ('mass', 'volume')
             AND comparable_value_match
        THEN 'product fields and quantity agree'
        WHEN product_status = 'partial'
             AND text_status = 'partial'
             AND text_category = 'energy'
             AND comparable_value_match
        THEN 'energy captured consistently but not used for consolidation'
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('count', 'packaging_only')
        THEN 'product fields used with descriptor from quantity'
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('mass', 'volume', 'multipack_measure', 'energy')
             AND NOT comparable_value_match
             AND NOT multipack_inner_value_match
        THEN 'conflict between product fields and quantity text'
        WHEN product_status = 'resolved'
        THEN 'structured product fields used'
        WHEN product_quantity_numeric = 0
             AND text_status IN ('resolved', 'partial')
             AND text_normalized_value IS NOT NULL
             AND coalesce(text_normalized_value, 0) > 0
        THEN 'structured zero ignored; quantity text used'
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND raw_numeric_matches_text
        THEN 'filled missing structure from quantity'
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND NOT raw_numeric_matches_text
        THEN 'structured numeric value disagrees with quantity text'
        WHEN text_status IS NOT NULL
        THEN text_note
        WHEN product_quantity_numeric IS NOT NULL AND product_base_unit IS NULL
        THEN 'structured numeric value present, but no usable unit found'
        ELSE 'no usable quantity signal found'
    END AS quantity_note
FROM staged;
'''

con.execute(quantity_cleaned_sql)

print("Reference cleaned view ready for internal change audit.")


In [ ]:

# This cell refreshes the helper setup for the current cleaner.
# DuckDB stays the main engine while this cell adds:
# - safe OCR cleanup before parsing
# - dataset-reviewed multilingual aliases
# - mixed-measure helpers for dual-unit and compound imperial expressions

from duckdb import sqltypes

FL_OZ_US_FACTOR = 29.5735295625
FL_OZ_IMPERIAL_FACTOR = 28.4130625

MEASURE_ALIASES = list(
    dict.fromkeys(
        MEASURE_ALIASES
        + [
            ("gr", "g", "mass", 1.0),
            ("mg", "g", "mass", 0.001),
            ("kilo", "g", "mass", 1000.0),
            ("pound", "g", "mass", 453.59237),
            ("pounds", "g", "mass", 453.59237),
        ]
    )
)

COUNT_DESCRIPTOR_ALIASES = list(
    dict.fromkeys(
        COUNT_DESCRIPTOR_ALIASES
        + [
            ("unit", "piece", "count"),
            ("units", "piece", "count"),
            ("unite", "piece", "count"),
            ("unites", "piece", "count"),
            ("unité", "piece", "count"),
            ("unités", "piece", "count"),
            ("oeuf", "piece", "count"),
            ("oeufs", "piece", "count"),
            ("egg", "piece", "count"),
            ("eggs", "piece", "count"),
            ("bar", "piece", "count"),
            ("bars", "piece", "count"),
            ("barre", "piece", "count"),
            ("barres", "piece", "count"),
            ("comprime", "tablet", "count"),
            ("comprimes", "tablet", "count"),
            ("caplet", "tablet", "count"),
            ("caplets", "tablet", "count"),
            ("softgel", "capsule", "count"),
            ("softgels", "capsule", "count"),
        ]
    )
)

APPROVED_MULTILINGUAL_ALIAS_TOKENS = [
    "bar",
    "barre",
    "barres",
    "boite",
    "boites",
    "caplet",
    "caplets",
    "chaque",
    "chacun",
    "chacune",
    "comprime",
    "comprimes",
    "egg",
    "eggs",
    "gr",
    "kilo",
    "lit",
    "morceau",
    "morceaux",
    "oeuf",
    "oeufs",
    "paquet",
    "paquets",
    "softgel",
    "softgels",
    "unite",
    "unites",
    "unit",
    "units",
    "unité",
    "unités",
]

STRUCTURAL_TAIL_TOKENS = ["each", "chacun", "chacune", "chaque"]
ALIAS_MINING_STOPWORDS = [
    "a",
    "an",
    "and",
    "au",
    "aux",
    "avec",
    "chaque",
    "chacun",
    "chacune",
    "d",
    "de",
    "des",
    "du",
    "each",
    "en",
    "et",
    "for",
    "l",
    "la",
    "le",
    "les",
    "net",
    "of",
    "ou",
    "par",
    "pour",
    "the",
    "wt",
    "x",
    "xl",
]


def normalize_sql(expr: str) -> str:
    normalized = f"lower(trim(coalesce({expr}, '')))"
    replacements = [
        ("’", "'"),
        ("â€™", "'"),
        ("`", "'"),
        ("Â´", "'"),
        ("×", "x"),
        ("Ã—", "x"),
        ("é", "e"),
        ("è", "e"),
        ("ê", "e"),
        ("ë", "e"),
        ("Ã©", "e"),
        ("Ã¨", "e"),
        ("Ãª", "e"),
        ("Ã«", "e"),
        ("à", "a"),
        ("â", "a"),
        ("ä", "a"),
        ("Ã ", "a"),
        ("Ã¢", "a"),
        ("Ã¤", "a"),
        ("î", "i"),
        ("ï", "i"),
        ("Ã®", "i"),
        ("Ã¯", "i"),
        ("ô", "o"),
        ("ö", "o"),
        ("Ã´", "o"),
        ("Ã¶", "o"),
        ("ù", "u"),
        ("û", "u"),
        ("ü", "u"),
        ("Ã¹", "u"),
        ("Ã»", "u"),
        ("Ã¼", "u"),
        ("ç", "c"),
        ("Ã§", "c"),
        ("œ", "oe"),
    ]
    for raw_text, canonical_text in replacements:
        normalized = f"replace({normalized}, {sql_quote(raw_text)}, {sql_quote(canonical_text)})"
    normalized = f"regexp_replace({normalized}, '\\s+', ' ', 'g')"
    return normalized


OCR_O_TOKEN_REGEX = re.compile(r"(?<![a-z0-9])([0-9o]+(?:[.,][0-9o]+)?|[.,][0-9o]+)(?![a-z])")
OCR_LEADING_IL_TOKEN_REGEX = re.compile(r"(?<![a-z0-9])([il][0-9]{2,}(?:[.,][0-9]+)?)(?=(?:\\s*(?:[a-z]+))?(?:\\b|$|[)\\],;:]))")


def _correct_o_numeric_token(token: str) -> str:
    corrected = token
    if corrected.startswith((".", ",")):
        corrected = "0" + corrected
    if re.fullmatch(r"[0-9o]+(?:[.,][0-9o]+)?", corrected):
        return corrected.replace("o", "0")
    return corrected


def _correct_leading_il_numeric_token(token: str) -> str:
    if re.fullmatch(r"[il][0-9]{2,}(?:[.,][0-9]+)?", token):
        return "1" + token[1:]
    return token


def safe_quantity_cleanup(text: str | None) -> str | None:
    if text is None:
        return None

    cleaned = re.sub(r"\s+", " ", text.strip())
    cleaned = cleaned.replace("fl. oz", "fl oz").replace("fl.oz", "fl oz")
    cleaned = re.sub(r"(?<=\d)\.\s+(?=\d)", ".", cleaned)
    cleaned = re.sub(r"(?<![0-9])([.,][0-9]+)(?![a-z])", lambda match: "0" + match.group(1), cleaned)
    cleaned = OCR_O_TOKEN_REGEX.sub(lambda match: _correct_o_numeric_token(match.group(1)), cleaned)
    cleaned = OCR_LEADING_IL_TOKEN_REGEX.sub(lambda match: _correct_leading_il_numeric_token(match.group(1)), cleaned)
    cleaned = re.sub(r"(?<=\w)[\.;,:]+$", "", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned


def has_ocr_like_numeric_token(text: str | None) -> bool:
    if text is None:
        return False
    for match in OCR_O_TOKEN_REGEX.finditer(text):
        token = match.group(1)
        if _correct_o_numeric_token(token) != token:
            return True
    for match in OCR_LEADING_IL_TOKEN_REGEX.finditer(text):
        token = match.group(1)
        if _correct_leading_il_numeric_token(token) != token:
            return True
    return False


def has_non_ascii_alpha(text: str | None) -> bool:
    if text is None:
        return False
    return any(character.isalpha() and ord(character) > 127 for character in text)


MEASURE_LOOKUP = {
    token: {"base_unit": base_unit, "category": category, "factor": factor}
    for token, base_unit, category, factor in MEASURE_ALIASES
}
METRIC_MASS_TOKENS = {
    "g",
    "gr",
    "gram",
    "grams",
    "gramme",
    "grammes",
    "mg",
    "kg",
    "kilo",
    "kilogram",
    "kilograms",
    "kilogramme",
    "kilogrammes",
}
METRIC_VOLUME_TOKENS = {
    "ml",
    "millilitre",
    "millilitres",
    "milliliter",
    "milliliters",
    "l",
    "lit",
    "litre",
    "litres",
    "liter",
    "liters",
    "cl",
    "dl",
}
IMPERIAL_MASS_TOKENS = {"oz", "lb", "lbs", "pound", "pounds"}

DESCRIPTOR_LOOKUP: dict[str, dict[str, str]] = {}
PY_MEASURE_OCCURRENCE_REGEX = None
PY_PLAIN_MULTIPACK_ANY_REGEX = None
PY_DESCRIPTOR_MULTIPACK_ANY_REGEX = None
PY_CONTEXTUAL_MIXED_KEYWORD_REGEX = None


def parse_numeric_text(text: str | None) -> float | None:
    if text is None:
        return None
    cleaned = text.strip().replace(",", ".")
    if cleaned.startswith("."):
        cleaned = "0" + cleaned
    try:
        return float(cleaned)
    except ValueError:
        return None


def decimal_places(text: str | None) -> int:
    if text is None:
        return 0
    cleaned = text.strip().replace(",", ".")
    if "." not in cleaned:
        return 0
    return len(cleaned.rsplit(".", 1)[1])


def to_base_value(value: float, unit: str, system: str = "default") -> tuple[float | None, str | None, str | None]:
    if unit == "fl oz":
        factor = FL_OZ_IMPERIAL_FACTOR if system == "imperial" else FL_OZ_US_FACTOR
        return value * factor, "ml", "volume"
    info = MEASURE_LOOKUP.get(unit)
    if info is None:
        return None, None, None
    return value * info["factor"], info["base_unit"], info["category"]


def rounding_half_step(value_text: str, unit: str, system: str = "default") -> float:
    decimals = decimal_places(value_text)
    step = 10 ** (-decimals)
    if unit == "fl oz":
        factor = FL_OZ_IMPERIAL_FACTOR if system == "imperial" else FL_OZ_US_FACTOR
    else:
        info = MEASURE_LOOKUP.get(unit)
        factor = 1.0 if info is None else info["factor"]
    return 0.5 * step * factor


def choose_metric_canonical(measures: list[dict], category: str, system: str = "default") -> tuple[float | None, str | None]:
    preferred_tokens = METRIC_MASS_TOKENS if category == "mass" else METRIC_VOLUME_TOKENS
    for measure in measures:
        if measure["unit"] in preferred_tokens:
            normalized_value, base_unit, _ = to_base_value(measure["value"], measure["unit"], system)
            return normalized_value, base_unit
    first = measures[0]
    normalized_value, base_unit, _ = to_base_value(first["value"], first["unit"], system)
    return normalized_value, base_unit


def equivalent_pair_result(measures: list[dict]) -> dict | None:
    first, second = measures
    if first["category"] != second["category"] or first["category"] not in {"mass", "volume"}:
        return None

    candidate_systems = ["default"]
    if first["category"] == "volume" and "fl oz" in {first["unit"], second["unit"]}:
        if any(unit in METRIC_VOLUME_TOKENS for unit in {first["unit"], second["unit"]}):
            candidate_systems = ["us", "imperial"]

    pair_units = {first["unit"], second["unit"]}
    metric_imperial_mass_pair = (
        first["category"] == "mass"
        and bool(pair_units & METRIC_MASS_TOKENS)
        and bool(pair_units & IMPERIAL_MASS_TOKENS)
    )

    matches: list[dict] = []
    for system in candidate_systems:
        value_1, unit_1, _ = to_base_value(first["value"], first["unit"], system)
        value_2, unit_2, _ = to_base_value(second["value"], second["unit"], system)
        if value_1 is None or value_2 is None or unit_1 != unit_2:
            continue
        relative_allowance = (
            0.045 if metric_imperial_mass_pair else VALUE_MATCH_TOLERANCE
        ) * max(abs(value_1), abs(value_2))
        rounding_allowance = (
            rounding_half_step(first["value_text"], first["unit"], system)
            + rounding_half_step(second["value_text"], second["unit"], system)
        )
        if metric_imperial_mass_pair:
            allowance = max(0.01, relative_allowance) + min(rounding_allowance, 1.0)
        else:
            allowance = max(0.01, relative_allowance, rounding_allowance)
        difference = abs(value_1 - value_2)
        if difference <= allowance:
            normalized_value, normalized_unit = choose_metric_canonical(measures, first["category"], system)
            matches.append(
                {
                    "status": "dual_unit_equivalent",
                    "category": first["category"],
                    "normalized_value": normalized_value,
                    "normalized_unit": normalized_unit,
                    "note": "equivalent dual-unit text normalized",
                    "system_used": system,
                    "difference": difference,
                }
            )

    if not matches:
        return None
    best_match = min(matches, key=lambda item: item["difference"])
    best_match.pop("difference", None)
    return best_match


def compound_imperial_result(measures: list[dict]) -> dict | None:
    categories = {measure["category"] for measure in measures}
    if categories != {"mass"}:
        return None

    metric_measures = [measure for measure in measures if measure["unit"] in METRIC_MASS_TOKENS]
    imperial_measures = [measure for measure in measures if measure["unit"] in IMPERIAL_MASS_TOKENS]
    if len(metric_measures) != 1 or not imperial_measures:
        return None

    metric_measure = metric_measures[0]
    metric_total, metric_unit, _ = to_base_value(metric_measure["value"], metric_measure["unit"])
    individually_equivalent_imperial: list[dict] = []
    remaining_imperial: list[dict] = []
    for measure in imperial_measures:
        converted_value, _, _ = to_base_value(measure["value"], measure["unit"])
        if converted_value is None:
            return None
        direct_allowance = max(
            0.01,
            VALUE_MATCH_TOLERANCE * max(abs(metric_total), abs(converted_value)),
        )
        if abs(metric_total - converted_value) <= direct_allowance:
            individually_equivalent_imperial.append(measure)
        else:
            remaining_imperial.append(measure)

    if remaining_imperial:
        imperial_total = 0.0
        imperial_allowance = 0.0
        for measure in remaining_imperial:
            converted_value, _, _ = to_base_value(measure["value"], measure["unit"])
            if converted_value is None:
                return None
            imperial_total += converted_value
            imperial_allowance += rounding_half_step(measure["value_text"], measure["unit"])

        allowance = max(
            0.01,
            VALUE_MATCH_TOLERANCE * max(abs(metric_total), abs(imperial_total)),
        )
        if abs(metric_total - imperial_total) > allowance:
            return {
                "status": "mixed_conflict",
                "category": "mixed_measure",
                "note": "mixed expression contradictory",
                "system_used": "default",
            }
    elif not individually_equivalent_imperial:
        return None

    return {
        "status": "compound_imperial_equivalent",
        "category": "mass",
        "normalized_value": metric_total,
        "normalized_unit": metric_unit,
        "note": "compound imperial text normalized against metric total",
        "system_used": "default",
    }


def resolve_mixed_measure_json(
    value_1_text: str | None,
    unit_1: str | None,
    value_2_text: str | None,
    unit_2: str | None,
    value_3_text: str | None,
    unit_3: str | None,
    value_4_text: str | None,
    unit_4: str | None,
) -> str:
    measures: list[dict] = []
    for value_text, unit in [
        (value_1_text, unit_1),
        (value_2_text, unit_2),
        (value_3_text, unit_3),
        (value_4_text, unit_4),
    ]:
        if value_text is None or unit is None:
            continue
        value = parse_numeric_text(value_text)
        if value is None:
            return json.dumps(
                {
                    "status": "mixed_unresolved",
                    "category": "mixed_measure",
                    "note": "mixed expression not safely reducible",
                    "system_used": "default",
                }
            )
        _, _, category = to_base_value(value, unit)
        if category is None:
            return json.dumps(
                {
                    "status": "mixed_unresolved",
                    "category": "mixed_measure",
                    "note": "mixed expression not safely reducible",
                    "system_used": "default",
                }
            )
        measures.append(
            {
                "value": value,
                "value_text": value_text,
                "unit": unit,
                "category": category,
            }
        )

    if len(measures) < 2:
        return json.dumps({})

    if len(measures) == 2:
        pair_result = equivalent_pair_result(measures)
        if pair_result is not None:
            return json.dumps(pair_result)
        if len({measure["category"] for measure in measures}) == 1 and measures[0]["category"] in {"mass", "volume"}:
            return json.dumps(
                {
                    "status": "mixed_conflict",
                    "category": "mixed_measure",
                    "note": "mixed expression contradictory",
                    "system_used": "default",
                }
            )
        return json.dumps(
            {
                "status": "mixed_unresolved",
                "category": "mixed_measure",
                "note": "mixed expression not safely reducible",
                "system_used": "default",
            }
        )

    compound_result = compound_imperial_result(measures)
    if compound_result is not None:
        return json.dumps(compound_result)

    if len({measure["category"] for measure in measures}) > 1:
        return json.dumps(
            {
                "status": "mixed_unresolved",
                "category": "mixed_measure",
                "note": "mixed expression not safely reducible",
                "system_used": "default",
            }
        )

    return json.dumps(
        {
            "status": "mixed_conflict",
            "category": "mixed_measure",
            "note": "mixed expression contradictory",
            "system_used": "default",
        }
    )


def normalized_values_close(value_1: float | None, value_2: float | None, base_unit: str | None) -> bool:
    if value_1 is None or value_2 is None:
        return False
    absolute_allowance = 1.0 if base_unit in {"g", "ml"} else 0.01
    return abs(value_1 - value_2) <= max(
        absolute_allowance,
        VALUE_MATCH_TOLERANCE * max(abs(value_1), abs(value_2)),
    )


def parse_count_token(text: str | None) -> float | None:
    if text is None:
        return None
    lowered = text.strip().lower()
    if lowered in {"un", "une", "a", "an"}:
        return 1.0
    return parse_numeric_text(lowered)


def descriptor_from_phrase(text: str | None) -> tuple[str | None, str | None]:
    if text is None:
        return None, None
    tokens = re.findall(r"[a-z]+(?:[-/][a-z]+)*", text.lower())
    for token in reversed(tokens):
        descriptor_info = DESCRIPTOR_LOOKUP.get(token)
        if descriptor_info is not None:
            return descriptor_info["item_descriptor"], descriptor_info["quantity_category"]
    return None, None


def extract_measure_occurrences(text: str) -> list[dict]:
    if PY_MEASURE_OCCURRENCE_REGEX is None:
        return []
    measures: list[dict] = []
    for match in PY_MEASURE_OCCURRENCE_REGEX.finditer(text):
        value_text = match.group(1)
        unit = match.group(2)
        value = parse_numeric_text(value_text)
        if value is None:
            continue
        normalized_value, base_unit, category = to_base_value(value, unit)
        if normalized_value is None or base_unit is None or category is None:
            continue
        measures.append(
            {
                "value_text": value_text,
                "value": value,
                "unit": unit,
                "normalized_value": normalized_value,
                "base_unit": base_unit,
                "category": category,
            }
        )
    return measures


def build_multipack_occurrence(
    count_text: str | None,
    descriptor_text: str | None,
    inner_value_text: str | None,
    unit_token: str | None,
) -> dict | None:
    count_numeric = parse_count_token(count_text)
    inner_value = parse_numeric_text(inner_value_text)
    if count_numeric is None or count_numeric <= 1 or inner_value is None or unit_token is None:
        return None
    inner_normalized_value, base_unit, category = to_base_value(inner_value, unit_token)
    if inner_normalized_value is None or base_unit is None or category not in {"mass", "volume"}:
        return None
    item_descriptor, descriptor_category = descriptor_from_phrase(descriptor_text)
    return {
        "count_numeric": count_numeric,
        "inner_normalized_value": inner_normalized_value,
        "total_normalized_value": count_numeric * inner_normalized_value,
        "base_unit": base_unit,
        "category": category,
        "item_descriptor": item_descriptor,
        "descriptor_category": descriptor_category,
    }


def resolve_complex_quantity_json(text: str | None) -> str:
    if text is None or text == "":
        return json.dumps({})

    measures = extract_measure_occurrences(text)
    contextual_mixed = bool(PY_CONTEXTUAL_MIXED_KEYWORD_REGEX and PY_CONTEXTUAL_MIXED_KEYWORD_REGEX.search(text))

    descriptor_occurrences: list[dict] = []
    occupied_spans: list[tuple[int, int]] = []
    if PY_DESCRIPTOR_MULTIPACK_ANY_REGEX is not None:
        for match in PY_DESCRIPTOR_MULTIPACK_ANY_REGEX.finditer(text):
            occurrence = build_multipack_occurrence(match.group(1), match.group(2), match.group(3), match.group(4))
            if occurrence is None:
                continue
            descriptor_occurrences.append(occurrence)
            occupied_spans.append(match.span())

    occurrences = descriptor_occurrences[:]
    if PY_PLAIN_MULTIPACK_ANY_REGEX is not None:
        for match in PY_PLAIN_MULTIPACK_ANY_REGEX.finditer(text):
            start, end = match.span()
            if any(start >= span_start and end <= span_end for span_start, span_end in occupied_spans):
                continue
            occurrence = build_multipack_occurrence(match.group(1), None, match.group(2), match.group(3))
            if occurrence is None:
                continue
            occurrences.append(occurrence)

    if occurrences:
        primary = occurrences[0]
        consistent_totals = all(
            occurrence["base_unit"] == primary["base_unit"]
            and occurrence["category"] == primary["category"]
            and normalized_values_close(
                occurrence["total_normalized_value"],
                primary["total_normalized_value"],
                primary["base_unit"],
            )
            for occurrence in occurrences
        )
        if not consistent_totals:
            return json.dumps(
                {
                    "status": "mixed_conflict",
                    "category": "mixed_measure",
                    "note": "mixed expression contradictory",
                }
            )

        same_pack_structure = all(
            normalized_values_close(
                occurrence["count_numeric"],
                primary["count_numeric"],
                None,
            )
            and normalized_values_close(
                occurrence["inner_normalized_value"],
                primary["inner_normalized_value"],
                primary["base_unit"],
            )
            for occurrence in occurrences
        )
        if len(occurrences) > 1 and not same_pack_structure:
            return json.dumps(
                {
                    "status": "contextual_mixed_unresolved" if contextual_mixed else "mixed_unresolved",
                    "category": "mixed_measure",
                    "note": "mixed expression not safely reducible",
                }
            )

        inner_values = [occurrence["inner_normalized_value"] for occurrence in occurrences]
        has_matching_total_measure = False
        has_conflicting_same_category_measure = False

        for measure in measures:
            if measure["base_unit"] != primary["base_unit"] or measure["category"] != primary["category"]:
                continue
            if any(
                normalized_values_close(measure["normalized_value"], inner_value, primary["base_unit"])
                for inner_value in inner_values
            ):
                continue
            if normalized_values_close(
                measure["normalized_value"],
                primary["total_normalized_value"],
                primary["base_unit"],
            ):
                has_matching_total_measure = True
            else:
                has_conflicting_same_category_measure = True

        if has_conflicting_same_category_measure:
            return json.dumps(
                {
                    "status": "contextual_mixed_unresolved" if contextual_mixed else "mixed_conflict",
                    "category": "mixed_measure",
                    "note": "mixed expression not safely reducible" if contextual_mixed else "mixed expression contradictory",
                }
            )

        note = "embedded multipack resolved from quantity text"
        if has_matching_total_measure:
            note = "embedded multipack total confirmed by surrounding text"
        if primary["item_descriptor"] is not None:
            note = "embedded descriptor multipack resolved from quantity text"
            if has_matching_total_measure:
                note = "embedded descriptor multipack total confirmed by surrounding text"

        return json.dumps(
            {
                "status": "embedded_descriptor_multipack" if primary["item_descriptor"] is not None else "embedded_plain_multipack",
                "category": "multipack_measure",
                "normalized_value": primary["total_normalized_value"],
                "normalized_unit": primary["base_unit"],
                "inner_normalized_value": primary["inner_normalized_value"],
                "pack_count": primary["count_numeric"],
                "item_descriptor": primary["item_descriptor"],
                "note": note,
            }
        )

    if len(measures) >= 2 and contextual_mixed:
        return json.dumps(
            {
                "status": "contextual_mixed_unresolved",
                "category": "mixed_measure",
                "note": "mixed expression not safely reducible",
            }
        )

    return json.dumps({})


for function_name in [
    "safe_quantity_cleanup",
    "has_ocr_like_numeric_token",
    "has_non_ascii_alpha",
    "resolve_complex_quantity_json",
    "resolve_mixed_measure_json",
]:
    try:
        con.remove_function(function_name)
    except Exception:
        pass

con.create_function(
    "safe_quantity_cleanup",
    safe_quantity_cleanup,
    [sqltypes.VARCHAR],
    sqltypes.VARCHAR,
)
con.create_function(
    "has_ocr_like_numeric_token",
    has_ocr_like_numeric_token,
    [sqltypes.VARCHAR],
    sqltypes.BOOLEAN,
)
con.create_function(
    "has_non_ascii_alpha",
    has_non_ascii_alpha,
    [sqltypes.VARCHAR],
    sqltypes.BOOLEAN,
)
con.create_function(
    "resolve_complex_quantity_json",
    resolve_complex_quantity_json,
    [sqltypes.VARCHAR],
    sqltypes.VARCHAR,
    null_handling="special",
)
con.create_function(
    "resolve_mixed_measure_json",
    resolve_mixed_measure_json,
    [
        sqltypes.VARCHAR,
        sqltypes.VARCHAR,
        sqltypes.VARCHAR,
        sqltypes.VARCHAR,
        sqltypes.VARCHAR,
        sqltypes.VARCHAR,
        sqltypes.VARCHAR,
        sqltypes.VARCHAR,
    ],
    sqltypes.VARCHAR,
    null_handling="special",
)

descriptor_rows = COUNT_DESCRIPTOR_ALIASES + PACKAGING_DESCRIPTOR_ALIASES
DESCRIPTOR_LOOKUP = {
    token: {"item_descriptor": item_descriptor, "quantity_category": quantity_category}
    for token, item_descriptor, quantity_category in descriptor_rows
}

con.execute(
    f'''
    CREATE OR REPLACE TEMP VIEW measure_aliases AS
    SELECT *
    FROM (VALUES
    {values_sql(MEASURE_ALIASES)}
    ) AS alias_rows(token, base_unit, quantity_category, factor);
    '''
)

con.execute(
    f'''
    CREATE OR REPLACE TEMP VIEW descriptor_aliases AS
    SELECT *
    FROM (VALUES
    {values_sql(descriptor_rows)}
    ) AS alias_rows(token, item_descriptor, quantity_category);
    '''
)

con.execute(
    f'''
    CREATE OR REPLACE TEMP VIEW household_aliases AS
    SELECT *
    FROM (VALUES
    {values_sql(HOUSEHOLD_ALIASES)}
    ) AS alias_rows(token, item_descriptor);
    '''
)

NUMBER_PATTERN = r"(?:[0-9]+(?:[.,][0-9]+)?|[.,][0-9]+)"
FRACTION_OR_NUMBER_PATTERN = rf"(?:[0-9]+/[0-9]+|{NUMBER_PATTERN})"
FREE_DESCRIPTOR_TOKEN_PATTERN = r"[a-z]+(?:[-/][a-z]+)*"
FREE_DESCRIPTOR_PHRASE_PATTERN = rf"{FREE_DESCRIPTOR_TOKEN_PATTERN}(?:\s+{FREE_DESCRIPTOR_TOKEN_PATTERN}){{0,2}}"
MEASURE_PATTERN = build_token_pattern([row[0] for row in MEASURE_ALIASES])
COUNT_DESCRIPTOR_PATTERN = build_token_pattern([row[0] for row in COUNT_DESCRIPTOR_ALIASES])
PACKAGING_DESCRIPTOR_PATTERN = build_token_pattern([row[0] for row in PACKAGING_DESCRIPTOR_ALIASES])
DESCRIPTOR_PATTERN = build_token_pattern([row[0] for row in descriptor_rows])
HOUSEHOLD_PATTERN = build_token_pattern([row[0] for row in HOUSEHOLD_ALIASES])
PLACEHOLDER_PATTERN = build_token_pattern(PLACEHOLDER_TERMS)
STRUCTURAL_TAIL_PATTERN = build_token_pattern(STRUCTURAL_TAIL_TOKENS)
APPROVED_MULTILINGUAL_PATTERN = build_token_pattern(APPROVED_MULTILINGUAL_ALIAS_TOKENS)

MEASURE_ANY_REGEX = rf"\b{MEASURE_PATTERN}\b"
HOUSEHOLD_ANY_REGEX = rf"\b{HOUSEHOLD_PATTERN}\b"
PLACEHOLDER_REGEX = rf"^(?:{PLACEHOLDER_PATTERN}|\?+)$"
SIMPLE_MEASURE_REGEX = rf"^\s*({NUMBER_PATTERN})\s*({MEASURE_PATTERN})\s*[.]?\s*$"
MULTIPACK_MEASURE_REGEX = rf"^\s*({NUMBER_PATTERN})\s*[*x]\s*({NUMBER_PATTERN})\s*({MEASURE_PATTERN})\b(?:\s*(?:{STRUCTURAL_TAIL_PATTERN}))?\s*[.]?\s*$"
DESCRIPTOR_MULTIPACK_REGEX = rf"^\s*({NUMBER_PATTERN}|un|une|a|an)\s+({FREE_DESCRIPTOR_PHRASE_PATTERN})\b\s*(?:de|of|[*x])\s*({NUMBER_PATTERN})\s*({MEASURE_PATTERN})\b(?:\s*(?:{STRUCTURAL_TAIL_PATTERN}))?\s*[.]?\s*$"
TOTAL_PLUS_COUNT_REGEX = rf"^\s*({NUMBER_PATTERN})\s*({MEASURE_PATTERN})\b\s*(?:/|-|,)\s*({NUMBER_PATTERN}|un|une|a|an)\s*({DESCRIPTOR_PATTERN})\b(?:\s*(?:{STRUCTURAL_TAIL_PATTERN}))?\s*[.]?\s*$"
TOTAL_PLUS_INNER_REGEX = rf"^\s*({NUMBER_PATTERN})\s*({MEASURE_PATTERN})\b\s*(?:/|-|,)\s*({NUMBER_PATTERN}|un|une|a|an)\s*({FREE_DESCRIPTOR_PHRASE_PATTERN})\b\s*(?:de|of|[*x])\s*({NUMBER_PATTERN})\s*({MEASURE_PATTERN})\b(?:\s*(?:{STRUCTURAL_TAIL_PATTERN}))?\s*[.]?\s*$"
DESCRIPTOR_WITH_MEASURE_REGEX = rf"^\s*({NUMBER_PATTERN}|un|une|a|an)?\s*({DESCRIPTOR_PATTERN})\b.*?({NUMBER_PATTERN})\s*({MEASURE_PATTERN})\b"
DESCRIPTOR_ONLY_REGEX = rf"^\s*({NUMBER_PATTERN}|un|une|a|an)?\s*({DESCRIPTOR_PATTERN})\b.*$"
PER_PACKAGING_REGEX = rf"^\s*({NUMBER_PATTERN})\s+par\s+({PACKAGING_DESCRIPTOR_PATTERN})\b"
HOUSEHOLD_REGEX = rf"^\s*({FRACTION_OR_NUMBER_PATTERN})\s*({HOUSEHOLD_PATTERN})\b"
NUMBER_ONLY_REGEX = rf"^\s*({NUMBER_PATTERN})\s*$"
MEASURE_VALUE_UNIT_REGEX = rf"({NUMBER_PATTERN})\s*({MEASURE_PATTERN})\b"

APPROVED_MULTILINGUAL_REGEX = rf"\b{APPROVED_MULTILINGUAL_PATTERN}\b"
CONTEXTUAL_MIXED_ANY_REGEX = r"(?:/ch\b|\b(?:capacity|capacite|ch|dr|drain|drained|dw|egoutte|egouttee|each|ea|net|par|per|poids|portion|qt|serving|servings|scoop|scoops|total|weight|wt)\b|\bu\b\s*[*x])"
PY_MEASURE_OCCURRENCE_REGEX = re.compile(rf"({NUMBER_PATTERN})\s*({MEASURE_PATTERN})\b")
PY_PLAIN_MULTIPACK_ANY_REGEX = re.compile(
    rf"(?<!\w)({NUMBER_PATTERN}|un|une|a|an)\s*[*x]\s*({NUMBER_PATTERN})\s*({MEASURE_PATTERN})\b"
)
PY_DESCRIPTOR_MULTIPACK_ANY_REGEX = re.compile(
    rf"(?<!\w)({NUMBER_PATTERN}|un|une|a|an)\s+({FREE_DESCRIPTOR_PHRASE_PATTERN})\s*(?:[*x]|de|of)\s*({NUMBER_PATTERN})\s*({MEASURE_PATTERN})\b"
)
PY_CONTEXTUAL_MIXED_KEYWORD_REGEX = re.compile(
    r"(?:/ch\b|\b(?:capacity|capacite|ch|chacun|chacune|chaque|dr|drain|drained|dw|ea|egoutte|egouttee|each|net|par|per|poids|portion|qt|serving|servings|scoop|scoops|total|weight|wt)\b|\bu\b\s*[*x])"
)
ALIAS_MINING_STOPWORDS_SQL_LIST = ", ".join(sql_quote(token) for token in ALIAS_MINING_STOPWORDS)

print("Current helper setup loaded.")


In [ ]:

# This cell creates the current feature view that powers the cleaner.
# The rule order follows the agreed plan:
# 1. clean obvious OCR/noise
# 2. parse simple measures
# 3. parse plain multipacks
# 4. parse descriptor-based multipacks
# 5. parse total-plus-count and total-plus-inner mixed expressions
# 6. resolve dual-unit and compound imperial equivalents
# 7. keep only real contradictions as conflict

quantity_features_sql = f'''
CREATE OR REPLACE TEMP VIEW quantity_cleaning_features AS
WITH base AS (
    SELECT
        row_id,
        code,
        product_quantity_unit,
        {normalize_sql("product_quantity_unit")} AS product_quantity_unit_normalized,
        product_quantity,
        TRY_CAST(replace(NULLIF(trim(product_quantity), ''), ',', '.') AS DOUBLE) AS product_quantity_numeric,
        quantity,
        {normalize_sql("quantity")} AS quantity_normalized_raw
    FROM quantity_raw
),
prepared AS (
    SELECT
        *,
        safe_quantity_cleanup(quantity_normalized_raw) AS quantity_normalized,
        quantity_normalized_raw IS DISTINCT FROM safe_quantity_cleanup(quantity_normalized_raw) AS quantity_ocr_cleanup_applied,
        has_ocr_like_numeric_token(quantity_normalized_raw) AS quantity_has_ocr_like_numeric_token,
        has_non_ascii_alpha(quantity_normalized_raw) AS quantity_has_non_ascii_alpha
    FROM base
),
extracted AS (
    SELECT
        p.*,
        regexp_extract_all(coalesce(quantity_normalized, ''), '{MEASURE_VALUE_UNIT_REGEX}', 1) AS measure_value_texts,
        regexp_extract_all(coalesce(quantity_normalized, ''), '{MEASURE_VALUE_UNIT_REGEX}', 2) AS measure_unit_tokens,
        array_length(regexp_extract_all(coalesce(quantity_normalized, ''), '{MEASURE_VALUE_UNIT_REGEX}', 1)) AS measure_match_count,
        NULLIF(list_extract(regexp_extract_all(coalesce(quantity_normalized, ''), '{MEASURE_VALUE_UNIT_REGEX}', 1), 1), '') AS measure_value_1_text,
        NULLIF(list_extract(regexp_extract_all(coalesce(quantity_normalized, ''), '{MEASURE_VALUE_UNIT_REGEX}', 2), 1), '') AS measure_unit_1_token,
        NULLIF(list_extract(regexp_extract_all(coalesce(quantity_normalized, ''), '{MEASURE_VALUE_UNIT_REGEX}', 1), 2), '') AS measure_value_2_text,
        NULLIF(list_extract(regexp_extract_all(coalesce(quantity_normalized, ''), '{MEASURE_VALUE_UNIT_REGEX}', 2), 2), '') AS measure_unit_2_token,
        NULLIF(list_extract(regexp_extract_all(coalesce(quantity_normalized, ''), '{MEASURE_VALUE_UNIT_REGEX}', 1), 3), '') AS measure_value_3_text,
        NULLIF(list_extract(regexp_extract_all(coalesce(quantity_normalized, ''), '{MEASURE_VALUE_UNIT_REGEX}', 2), 3), '') AS measure_unit_3_token,
        NULLIF(list_extract(regexp_extract_all(coalesce(quantity_normalized, ''), '{MEASURE_VALUE_UNIT_REGEX}', 1), 4), '') AS measure_value_4_text,
        NULLIF(list_extract(regexp_extract_all(coalesce(quantity_normalized, ''), '{MEASURE_VALUE_UNIT_REGEX}', 2), 4), '') AS measure_unit_4_token,
        NULLIF(regexp_extract(quantity_normalized, '{MULTIPACK_MEASURE_REGEX}', 1), '') AS multipack_count_text,
        NULLIF(regexp_extract(quantity_normalized, '{MULTIPACK_MEASURE_REGEX}', 2), '') AS multipack_inner_value_text,
        NULLIF(regexp_extract(quantity_normalized, '{MULTIPACK_MEASURE_REGEX}', 3), '') AS multipack_unit_token,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_MULTIPACK_REGEX}', 1), '') AS descriptor_multipack_count_text,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_MULTIPACK_REGEX}', 2), '') AS descriptor_multipack_token,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_MULTIPACK_REGEX}', 3), '') AS descriptor_multipack_inner_value_text,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_MULTIPACK_REGEX}', 4), '') AS descriptor_multipack_unit_token,
        NULLIF(regexp_extract(quantity_normalized, '{TOTAL_PLUS_COUNT_REGEX}', 1), '') AS total_plus_value_text,
        NULLIF(regexp_extract(quantity_normalized, '{TOTAL_PLUS_COUNT_REGEX}', 2), '') AS total_plus_unit_token,
        NULLIF(regexp_extract(quantity_normalized, '{TOTAL_PLUS_COUNT_REGEX}', 3), '') AS total_plus_count_text,
        NULLIF(regexp_extract(quantity_normalized, '{TOTAL_PLUS_COUNT_REGEX}', 4), '') AS total_plus_descriptor_token,
        NULLIF(regexp_extract(quantity_normalized, '{TOTAL_PLUS_INNER_REGEX}', 1), '') AS total_plus_inner_total_value_text,
        NULLIF(regexp_extract(quantity_normalized, '{TOTAL_PLUS_INNER_REGEX}', 2), '') AS total_plus_inner_total_unit_token,
        NULLIF(regexp_extract(quantity_normalized, '{TOTAL_PLUS_INNER_REGEX}', 3), '') AS total_plus_inner_count_text,
        NULLIF(regexp_extract(quantity_normalized, '{TOTAL_PLUS_INNER_REGEX}', 4), '') AS total_plus_inner_descriptor_token,
        NULLIF(regexp_extract(quantity_normalized, '{TOTAL_PLUS_INNER_REGEX}', 5), '') AS total_plus_inner_value_text,
        NULLIF(regexp_extract(quantity_normalized, '{TOTAL_PLUS_INNER_REGEX}', 6), '') AS total_plus_inner_unit_token,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_WITH_MEASURE_REGEX}', 1), '') AS descriptor_measure_leading_token,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_WITH_MEASURE_REGEX}', 2), '') AS descriptor_measure_token,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_WITH_MEASURE_REGEX}', 3), '') AS descriptor_measure_value_text,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_WITH_MEASURE_REGEX}', 4), '') AS descriptor_measure_unit_token,
        NULLIF(regexp_extract(quantity_normalized, '{SIMPLE_MEASURE_REGEX}', 1), '') AS simple_value_text,
        NULLIF(regexp_extract(quantity_normalized, '{SIMPLE_MEASURE_REGEX}', 2), '') AS simple_unit_token,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_ONLY_REGEX}', 1), '') AS descriptor_only_leading_token,
        NULLIF(regexp_extract(quantity_normalized, '{DESCRIPTOR_ONLY_REGEX}', 2), '') AS descriptor_only_token,
        NULLIF(regexp_extract(quantity_normalized, '{PER_PACKAGING_REGEX}', 1), '') AS per_pack_count_text,
        NULLIF(regexp_extract(quantity_normalized, '{PER_PACKAGING_REGEX}', 2), '') AS per_pack_token,
        NULLIF(regexp_extract(quantity_normalized, '{HOUSEHOLD_REGEX}', 1), '') AS household_value_text,
        NULLIF(regexp_extract(quantity_normalized, '{HOUSEHOLD_REGEX}', 2), '') AS household_unit_token,
        regexp_matches(quantity_normalized, '{PLACEHOLDER_REGEX}') AS quantity_is_placeholder,
        regexp_matches(quantity_normalized, '{NUMBER_ONLY_REGEX}') AS quantity_is_number_only,
        regexp_matches(quantity_normalized, '{HOUSEHOLD_ANY_REGEX}') AS quantity_has_household_token,
        regexp_matches(quantity_normalized, '{MEASURE_ANY_REGEX}') AS quantity_has_measure_token,
        regexp_matches(quantity_normalized, '{CONTEXTUAL_MIXED_ANY_REGEX}') AS quantity_has_contextual_mixed_token,
        regexp_matches(quantity_normalized, '\d') AS quantity_has_digits,
        regexp_matches(quantity_normalized, '{APPROVED_MULTILINGUAL_REGEX}') AS quantity_uses_new_multilingual_alias
    FROM prepared p
),
joined AS (
    SELECT
        e.*,
        pm.base_unit AS product_base_unit,
        pm.quantity_category AS product_category,
        pm.factor AS product_factor,
        mm.base_unit AS multipack_base_unit,
        mm.quantity_category AS multipack_category,
        mm.factor AS multipack_factor,
        dpm.base_unit AS descriptor_multipack_base_unit,
        dpm.quantity_category AS descriptor_multipack_category,
        dpm.factor AS descriptor_multipack_factor,
        tm.base_unit AS total_plus_base_unit,
        tm.quantity_category AS total_plus_measure_category,
        tm.factor AS total_plus_factor,
        tim_total.base_unit AS total_plus_inner_total_base_unit,
        tim_total.quantity_category AS total_plus_inner_total_category,
        tim_total.factor AS total_plus_inner_total_factor,
        tim_inner.base_unit AS total_plus_inner_base_unit,
        tim_inner.quantity_category AS total_plus_inner_category,
        tim_inner.factor AS total_plus_inner_factor,
        dm.base_unit AS descriptor_measure_base_unit,
        dm.quantity_category AS descriptor_measure_category,
        dm.factor AS descriptor_measure_factor,
        sm.base_unit AS simple_base_unit,
        sm.quantity_category AS simple_category,
        sm.factor AS simple_factor,
        dpm_desc.item_descriptor AS descriptor_multipack_item_descriptor,
        tpd.item_descriptor AS total_plus_item_descriptor,
        tpd.quantity_category AS total_plus_descriptor_category,
        tpi.item_descriptor AS total_plus_inner_item_descriptor,
        tpi.quantity_category AS total_plus_inner_descriptor_category,
        de.item_descriptor AS descriptor_measure_item_descriptor,
        de.quantity_category AS descriptor_measure_descriptor_category,
        d0.item_descriptor AS descriptor_only_item_descriptor,
        d0.quantity_category AS descriptor_only_category,
        pp.item_descriptor AS per_pack_item_descriptor,
        hh.item_descriptor AS household_descriptor
    FROM extracted e
    LEFT JOIN measure_aliases pm ON e.product_quantity_unit_normalized = pm.token
    LEFT JOIN measure_aliases mm ON e.multipack_unit_token = mm.token
    LEFT JOIN measure_aliases dpm ON e.descriptor_multipack_unit_token = dpm.token
    LEFT JOIN measure_aliases tm ON e.total_plus_unit_token = tm.token
    LEFT JOIN measure_aliases tim_total ON e.total_plus_inner_total_unit_token = tim_total.token
    LEFT JOIN measure_aliases tim_inner ON e.total_plus_inner_unit_token = tim_inner.token
    LEFT JOIN measure_aliases dm ON e.descriptor_measure_unit_token = dm.token
    LEFT JOIN measure_aliases sm ON e.simple_unit_token = sm.token
    LEFT JOIN descriptor_aliases dpm_desc ON e.descriptor_multipack_token = dpm_desc.token
    LEFT JOIN descriptor_aliases tpd ON e.total_plus_descriptor_token = tpd.token
    LEFT JOIN descriptor_aliases tpi ON e.total_plus_inner_descriptor_token = tpi.token
    LEFT JOIN descriptor_aliases de ON e.descriptor_measure_token = de.token
    LEFT JOIN descriptor_aliases d0 ON e.descriptor_only_token = d0.token
    LEFT JOIN descriptor_aliases pp ON e.per_pack_token = pp.token
    LEFT JOIN household_aliases hh ON e.household_unit_token = hh.token
),
typed AS (
    SELECT
        *,
        CASE
            WHEN lower(coalesce(multipack_count_text, '')) IN ('un', 'une', 'a', 'an') THEN 1.0
            ELSE TRY_CAST(replace(multipack_count_text, ',', '.') AS DOUBLE)
        END AS multipack_count_numeric,
        TRY_CAST(replace(multipack_inner_value_text, ',', '.') AS DOUBLE) AS multipack_inner_value_numeric,
        CASE
            WHEN lower(coalesce(descriptor_multipack_count_text, '')) IN ('un', 'une', 'a', 'an') THEN 1.0
            ELSE TRY_CAST(replace(descriptor_multipack_count_text, ',', '.') AS DOUBLE)
        END AS descriptor_multipack_count_numeric,
        TRY_CAST(replace(descriptor_multipack_inner_value_text, ',', '.') AS DOUBLE) AS descriptor_multipack_inner_value_numeric,
        TRY_CAST(replace(total_plus_value_text, ',', '.') AS DOUBLE) AS total_plus_value_numeric,
        CASE
            WHEN lower(coalesce(total_plus_count_text, '')) IN ('un', 'une', 'a', 'an') THEN 1.0
            ELSE TRY_CAST(replace(total_plus_count_text, ',', '.') AS DOUBLE)
        END AS total_plus_count_numeric,
        TRY_CAST(replace(total_plus_inner_total_value_text, ',', '.') AS DOUBLE) AS total_plus_inner_total_value_numeric,
        CASE
            WHEN lower(coalesce(total_plus_inner_count_text, '')) IN ('un', 'une', 'a', 'an') THEN 1.0
            ELSE TRY_CAST(replace(total_plus_inner_count_text, ',', '.') AS DOUBLE)
        END AS total_plus_inner_count_numeric,
        TRY_CAST(replace(total_plus_inner_value_text, ',', '.') AS DOUBLE) AS total_plus_inner_value_numeric,
        CASE
            WHEN lower(coalesce(descriptor_measure_leading_token, '')) IN ('un', 'une', 'a', 'an') THEN 1.0
            ELSE TRY_CAST(replace(descriptor_measure_leading_token, ',', '.') AS DOUBLE)
        END AS descriptor_measure_count_numeric,
        TRY_CAST(replace(descriptor_measure_value_text, ',', '.') AS DOUBLE) AS descriptor_measure_value_numeric,
        TRY_CAST(replace(simple_value_text, ',', '.') AS DOUBLE) AS simple_value_numeric,
        TRY_CAST(replace(measure_value_1_text, ',', '.') AS DOUBLE) AS measure_1_numeric,
        TRY_CAST(replace(measure_value_2_text, ',', '.') AS DOUBLE) AS measure_2_numeric,
        TRY_CAST(replace(measure_value_3_text, ',', '.') AS DOUBLE) AS measure_3_numeric,
        TRY_CAST(replace(measure_value_4_text, ',', '.') AS DOUBLE) AS measure_4_numeric,
        CASE
            WHEN lower(coalesce(descriptor_only_leading_token, '')) IN ('un', 'une', 'a', 'an') THEN 1.0
            ELSE TRY_CAST(replace(descriptor_only_leading_token, ',', '.') AS DOUBLE)
        END AS descriptor_only_count_numeric,
        TRY_CAST(replace(per_pack_count_text, ',', '.') AS DOUBLE) AS per_pack_count_numeric,
        CASE
            WHEN household_value_text LIKE '%/%' THEN
                TRY_CAST(split_part(household_value_text, '/', 1) AS DOUBLE)
                / NULLIF(TRY_CAST(split_part(household_value_text, '/', 2) AS DOUBLE), 0)
            ELSE TRY_CAST(replace(household_value_text, ',', '.') AS DOUBLE)
        END AS household_value_numeric,
        CASE WHEN product_quantity_numeric IS NOT NULL AND product_factor IS NOT NULL THEN product_quantity_numeric * product_factor END AS product_normalized_value,
        CASE WHEN multipack_inner_value_numeric IS NOT NULL AND multipack_factor IS NOT NULL THEN multipack_inner_value_numeric * multipack_factor END AS multipack_inner_normalized_value,
        CASE WHEN multipack_count_numeric IS NOT NULL AND multipack_inner_value_numeric IS NOT NULL AND multipack_factor IS NOT NULL THEN multipack_count_numeric * multipack_inner_value_numeric * multipack_factor END AS multipack_normalized_value,
        CASE WHEN descriptor_multipack_count_numeric IS NOT NULL AND descriptor_multipack_count_numeric > 1 AND descriptor_multipack_inner_value_numeric IS NOT NULL AND descriptor_multipack_factor IS NOT NULL THEN descriptor_multipack_inner_value_numeric * descriptor_multipack_factor END AS descriptor_multipack_inner_normalized_value,
        CASE WHEN descriptor_multipack_count_numeric IS NOT NULL AND descriptor_multipack_count_numeric > 1 AND descriptor_multipack_inner_value_numeric IS NOT NULL AND descriptor_multipack_factor IS NOT NULL THEN descriptor_multipack_count_numeric * descriptor_multipack_inner_value_numeric * descriptor_multipack_factor END AS descriptor_multipack_total_normalized_value,
        CASE WHEN total_plus_value_numeric IS NOT NULL AND total_plus_factor IS NOT NULL THEN total_plus_value_numeric * total_plus_factor END AS total_plus_total_normalized_value,
        CASE WHEN total_plus_inner_total_value_numeric IS NOT NULL AND total_plus_inner_total_factor IS NOT NULL THEN total_plus_inner_total_value_numeric * total_plus_inner_total_factor END AS total_plus_inner_total_normalized_value,
        CASE WHEN total_plus_inner_value_numeric IS NOT NULL AND total_plus_inner_factor IS NOT NULL THEN total_plus_inner_value_numeric * total_plus_inner_factor END AS total_plus_inner_normalized_value,
        CASE WHEN total_plus_inner_count_numeric IS NOT NULL AND total_plus_inner_value_numeric IS NOT NULL AND total_plus_inner_factor IS NOT NULL THEN total_plus_inner_count_numeric * total_plus_inner_value_numeric * total_plus_inner_factor END AS total_plus_inner_derived_total_normalized_value,
        CASE WHEN descriptor_measure_value_numeric IS NOT NULL AND descriptor_measure_factor IS NOT NULL THEN descriptor_measure_value_numeric * descriptor_measure_factor END AS descriptor_measure_normalized_value,
        CASE WHEN simple_value_numeric IS NOT NULL AND simple_factor IS NOT NULL THEN simple_value_numeric * simple_factor END AS simple_normalized_value,
        resolve_mixed_measure_json(
            measure_value_1_text,
            measure_unit_1_token,
            measure_value_2_text,
            measure_unit_2_token,
            measure_value_3_text,
            measure_unit_3_token,
            measure_value_4_text,
            measure_unit_4_token
        ) AS mixed_measure_resolution_json,
        resolve_complex_quantity_json(quantity_normalized) AS complex_quantity_resolution_json
    FROM joined
),
calculated AS (
    SELECT
        *,
        NULLIF(json_extract_string(complex_quantity_resolution_json, '$.status'), '') AS complex_resolution_status,
        NULLIF(json_extract_string(complex_quantity_resolution_json, '$.category'), '') AS complex_resolution_category,
        TRY_CAST(json_extract_string(complex_quantity_resolution_json, '$.normalized_value') AS DOUBLE) AS complex_resolution_value,
        NULLIF(json_extract_string(complex_quantity_resolution_json, '$.normalized_unit'), '') AS complex_resolution_unit,
        TRY_CAST(json_extract_string(complex_quantity_resolution_json, '$.inner_normalized_value') AS DOUBLE) AS complex_resolution_inner_value,
        TRY_CAST(json_extract_string(complex_quantity_resolution_json, '$.pack_count') AS DOUBLE) AS complex_resolution_pack_count,
        NULLIF(json_extract_string(complex_quantity_resolution_json, '$.item_descriptor'), '') AS complex_resolution_item_descriptor,
        NULLIF(json_extract_string(complex_quantity_resolution_json, '$.note'), '') AS complex_resolution_note,
        NULLIF(json_extract_string(mixed_measure_resolution_json, '$.status'), '') AS mixed_resolution_status,
        NULLIF(json_extract_string(mixed_measure_resolution_json, '$.category'), '') AS mixed_resolution_category,
        TRY_CAST(json_extract_string(mixed_measure_resolution_json, '$.normalized_value') AS DOUBLE) AS mixed_resolution_value,
        NULLIF(json_extract_string(mixed_measure_resolution_json, '$.normalized_unit'), '') AS mixed_resolution_unit,
        NULLIF(json_extract_string(mixed_measure_resolution_json, '$.note'), '') AS mixed_resolution_note,
        NULLIF(json_extract_string(mixed_measure_resolution_json, '$.system_used'), '') AS mixed_resolution_system,
        CASE
            WHEN total_plus_inner_total_normalized_value IS NOT NULL
                 AND total_plus_inner_derived_total_normalized_value IS NOT NULL
                 AND total_plus_inner_total_base_unit = total_plus_inner_base_unit
            THEN abs(total_plus_inner_total_normalized_value - total_plus_inner_derived_total_normalized_value)
                 <= greatest(
                     0.01,
                     {VALUE_MATCH_TOLERANCE} * greatest(
                         abs(total_plus_inner_total_normalized_value),
                         abs(total_plus_inner_derived_total_normalized_value)
                     )
                 )
            ELSE FALSE
        END AS total_plus_inner_matches_total
    FROM typed
),
classified AS (
    SELECT
        *,
        CASE
            WHEN quantity IS NULL THEN NULL
            WHEN quantity_normalized = '' THEN 'blank'
            WHEN quantity_is_placeholder THEN 'placeholder_or_unknown'
            WHEN quantity_has_household_token THEN 'household_unit'
            WHEN simple_normalized_value IS NOT NULL AND simple_category IN ('mass', 'volume') THEN 'simple_measure'
            WHEN simple_normalized_value IS NOT NULL AND simple_category = 'energy' THEN 'simple_energy'
            WHEN multipack_normalized_value IS NOT NULL AND multipack_category IN ('mass', 'volume') THEN 'plain_multipack'
            WHEN descriptor_multipack_total_normalized_value IS NOT NULL AND descriptor_multipack_category IN ('mass', 'volume') THEN 'descriptor_multipack'
            WHEN total_plus_inner_matches_total THEN 'total_plus_inner_multipack'
            WHEN total_plus_total_normalized_value IS NOT NULL
                 AND total_plus_measure_category IN ('mass', 'volume')
                 AND total_plus_count_numeric IS NOT NULL
                 AND total_plus_descriptor_category IN ('count', 'packaging_only')
            THEN 'total_plus_count'
            WHEN complex_resolution_status IS NOT NULL THEN complex_resolution_status
            WHEN descriptor_measure_normalized_value IS NOT NULL AND descriptor_measure_category IN ('mass', 'volume') THEN 'descriptor_measure'
            WHEN descriptor_measure_normalized_value IS NOT NULL AND descriptor_measure_category = 'energy' THEN 'descriptor_measure_energy'
            WHEN descriptor_only_item_descriptor IS NOT NULL AND descriptor_only_category = 'count' THEN 'count_descriptor'
            WHEN descriptor_only_item_descriptor IS NOT NULL AND descriptor_only_category = 'packaging_only' THEN 'packaging_only'
            WHEN per_pack_item_descriptor IS NOT NULL THEN 'per_packaging'
            WHEN mixed_resolution_status = 'mixed_conflict' AND quantity_has_contextual_mixed_token THEN 'contextual_mixed_unresolved'
            WHEN mixed_resolution_status IS NOT NULL THEN mixed_resolution_status
            WHEN quantity_has_non_ascii_alpha THEN 'unsupported_foreign_unit'
            WHEN quantity_has_ocr_like_numeric_token THEN 'ocr_like_numeric_token'
            WHEN quantity_is_number_only THEN 'number_only'
            WHEN NOT quantity_has_digits THEN 'noise_or_non_quantity'
            WHEN quantity_has_measure_token THEN 'mixed_unresolved'
            ELSE 'unparsed'
        END AS text_subtype
    FROM calculated
)
SELECT
    *,
    CASE
        WHEN quantity IS NULL THEN NULL
        WHEN text_subtype IN (
            'plain_multipack',
            'descriptor_multipack',
            'embedded_plain_multipack',
            'embedded_descriptor_multipack',
            'total_plus_count',
            'total_plus_inner_multipack',
            'dual_unit_equivalent',
            'compound_imperial_equivalent',
            'descriptor_measure',
            'simple_measure',
            'count_descriptor'
        ) THEN 'resolved'
        WHEN text_subtype IN ('descriptor_measure_energy', 'simple_energy', 'packaging_only', 'per_packaging') THEN 'partial'
        WHEN text_subtype = 'mixed_conflict' THEN 'conflict'
        ELSE 'unresolved'
    END AS text_status,
    CASE
        WHEN quantity IS NULL THEN NULL
        WHEN text_subtype = 'placeholder_or_unknown' THEN 'placeholder_or_unknown'
        WHEN text_subtype = 'household_unit' THEN 'household_unit'
        WHEN text_subtype IN ('plain_multipack', 'descriptor_multipack', 'embedded_plain_multipack', 'embedded_descriptor_multipack', 'total_plus_count', 'total_plus_inner_multipack') THEN 'multipack_measure'
        WHEN text_subtype IN ('dual_unit_equivalent', 'compound_imperial_equivalent') THEN mixed_resolution_category
        WHEN text_subtype IN ('contextual_mixed_unresolved', 'mixed_conflict', 'mixed_unresolved') THEN 'mixed_measure'
        WHEN text_subtype IN ('descriptor_measure', 'simple_measure') THEN coalesce(descriptor_measure_category, simple_category)
        WHEN text_subtype IN ('descriptor_measure_energy', 'simple_energy') THEN 'energy'
        WHEN text_subtype = 'count_descriptor' THEN 'count'
        WHEN text_subtype IN ('packaging_only', 'per_packaging') THEN 'packaging_only'
        WHEN text_subtype = 'noise_or_non_quantity' THEN 'noise_or_non_quantity'
        ELSE NULL
    END AS text_category,
    CASE
        WHEN text_subtype = 'plain_multipack' THEN multipack_normalized_value
        WHEN text_subtype = 'descriptor_multipack' THEN descriptor_multipack_total_normalized_value
        WHEN text_subtype IN ('embedded_plain_multipack', 'embedded_descriptor_multipack') THEN complex_resolution_value
        WHEN text_subtype = 'total_plus_count' THEN total_plus_total_normalized_value
        WHEN text_subtype = 'total_plus_inner_multipack' THEN total_plus_inner_total_normalized_value
        WHEN text_subtype IN ('dual_unit_equivalent', 'compound_imperial_equivalent') THEN mixed_resolution_value
        WHEN text_subtype IN ('descriptor_measure', 'descriptor_measure_energy') THEN descriptor_measure_normalized_value
        WHEN text_subtype IN ('simple_measure', 'simple_energy') THEN simple_normalized_value
        WHEN text_subtype = 'count_descriptor' THEN coalesce(descriptor_only_count_numeric, 1.0)
        ELSE NULL
    END AS text_normalized_value,
    CASE
        WHEN text_subtype = 'plain_multipack' THEN multipack_base_unit
        WHEN text_subtype = 'descriptor_multipack' THEN descriptor_multipack_base_unit
        WHEN text_subtype IN ('embedded_plain_multipack', 'embedded_descriptor_multipack') THEN complex_resolution_unit
        WHEN text_subtype = 'total_plus_count' THEN total_plus_base_unit
        WHEN text_subtype = 'total_plus_inner_multipack' THEN total_plus_inner_total_base_unit
        WHEN text_subtype IN ('dual_unit_equivalent', 'compound_imperial_equivalent') THEN mixed_resolution_unit
        WHEN text_subtype IN ('descriptor_measure', 'descriptor_measure_energy') THEN descriptor_measure_base_unit
        WHEN text_subtype IN ('simple_measure', 'simple_energy') THEN simple_base_unit
        WHEN text_subtype = 'count_descriptor' THEN 'count'
        ELSE NULL
    END AS text_normalized_unit,
    CASE
        WHEN text_subtype = 'plain_multipack' THEN multipack_inner_normalized_value
        WHEN text_subtype = 'descriptor_multipack' THEN descriptor_multipack_inner_normalized_value
        WHEN text_subtype IN ('embedded_plain_multipack', 'embedded_descriptor_multipack') THEN complex_resolution_inner_value
        WHEN text_subtype = 'total_plus_inner_multipack' THEN total_plus_inner_normalized_value
        ELSE NULL
    END AS text_inner_normalized_value,
    CASE
        WHEN text_subtype = 'plain_multipack' THEN multipack_count_numeric
        WHEN text_subtype = 'descriptor_multipack' THEN descriptor_multipack_count_numeric
        WHEN text_subtype IN ('embedded_plain_multipack', 'embedded_descriptor_multipack') THEN complex_resolution_pack_count
        WHEN text_subtype = 'total_plus_count' THEN total_plus_count_numeric
        WHEN text_subtype = 'total_plus_inner_multipack' THEN total_plus_inner_count_numeric
        WHEN text_subtype = 'count_descriptor' THEN 1.0
        WHEN text_subtype IN ('packaging_only', 'per_packaging') THEN coalesce(descriptor_only_count_numeric, per_pack_count_numeric, 1.0)
        WHEN text_subtype IN ('descriptor_measure', 'descriptor_measure_energy', 'simple_measure', 'simple_energy', 'dual_unit_equivalent', 'compound_imperial_equivalent') THEN 1.0
        ELSE NULL
    END AS text_pack_count,
    CASE
        WHEN text_subtype = 'total_plus_count' THEN total_plus_item_descriptor
        WHEN text_subtype = 'total_plus_inner_multipack' THEN total_plus_inner_item_descriptor
        WHEN text_subtype = 'descriptor_multipack' THEN descriptor_multipack_item_descriptor
        WHEN text_subtype = 'embedded_descriptor_multipack' THEN complex_resolution_item_descriptor
        WHEN text_subtype IN ('descriptor_measure', 'descriptor_measure_energy') THEN descriptor_measure_item_descriptor
        WHEN text_subtype IN ('count_descriptor', 'packaging_only') THEN descriptor_only_item_descriptor
        WHEN text_subtype = 'per_packaging' THEN per_pack_item_descriptor
        ELSE NULL
    END AS text_item_descriptor,
    CASE
        WHEN quantity IS NULL THEN 'quantity missing'
        WHEN text_subtype = 'blank' THEN 'blank quantity string'
        WHEN text_subtype = 'placeholder_or_unknown' THEN 'placeholder or unknown text'
        WHEN text_subtype = 'household_unit' THEN 'household unit not used for consolidation'
        WHEN text_subtype = 'plain_multipack' THEN 'derived total from quantity multipack'
        WHEN text_subtype = 'descriptor_multipack' THEN 'derived total from quantity descriptor multipack'
        WHEN text_subtype IN ('embedded_plain_multipack', 'embedded_descriptor_multipack', 'contextual_mixed_unresolved') THEN complex_resolution_note
        WHEN text_subtype = 'total_plus_count' THEN 'total size plus count metadata from quantity'
        WHEN text_subtype = 'total_plus_inner_multipack' THEN 'explicit total matches descriptor-based inner multipack math'
        WHEN text_subtype = 'compound_imperial_equivalent' THEN 'compound imperial text normalized against metric total'
        WHEN text_subtype = 'dual_unit_equivalent' AND quantity_ocr_cleanup_applied THEN 'equivalent dual-unit text normalized after OCR cleanup'
        WHEN text_subtype = 'dual_unit_equivalent' THEN 'equivalent dual-unit text normalized'
        WHEN text_subtype = 'mixed_conflict' THEN 'mixed expression contradictory'
        WHEN text_subtype = 'mixed_unresolved' THEN 'mixed expression not safely reducible'
        WHEN text_subtype = 'descriptor_measure_energy' THEN 'energy text captured but not used for consolidation'
        WHEN text_subtype = 'descriptor_measure' AND quantity_ocr_cleanup_applied THEN 'descriptor plus measure from quantity after OCR cleanup'
        WHEN text_subtype = 'descriptor_measure' THEN 'descriptor plus measure from quantity'
        WHEN text_subtype = 'simple_energy' THEN 'energy text captured but not used for consolidation'
        WHEN text_subtype = 'simple_measure' AND quantity_ocr_cleanup_applied THEN 'simple measure from quantity after OCR cleanup'
        WHEN text_subtype = 'simple_measure' THEN 'simple measure from quantity'
        WHEN text_subtype = 'count_descriptor' THEN 'count descriptor from quantity'
        WHEN text_subtype = 'packaging_only' THEN 'packaging only, no comparable size'
        WHEN text_subtype = 'per_packaging' THEN 'per packaging phrase, no comparable size'
        WHEN text_subtype = 'unsupported_foreign_unit' THEN 'unsupported foreign unit token'
        WHEN text_subtype = 'ocr_like_numeric_token' THEN 'ocr-like numeric token'
        WHEN text_subtype = 'number_only' THEN 'number only, unit missing'
        WHEN text_subtype = 'noise_or_non_quantity' THEN 'non-quantity text in quantity field'
        ELSE 'unparsed quantity text'
    END AS text_note
FROM classified;
'''

con.execute(quantity_features_sql)

show_query(
    "Current parsed text signal summary",
    '''
    SELECT COALESCE(text_status, 'null') AS text_status,
           COALESCE(text_category, 'null') AS text_category,
           COALESCE(text_subtype, 'null') AS text_subtype,
           COUNT(*) AS rows
    FROM quantity_cleaning_features
    GROUP BY 1, 2, 3
    ORDER BY rows DESC, text_status, text_category, text_subtype
    '''
)


In [ ]:

# This cell applies the final cross-field decision rules and produces the single cleaned output view.
# It keeps one user-facing table named `quantity_cleaned` and one internal debug table.

quantity_cleaned_sql = f'''
CREATE OR REPLACE TEMP VIEW quantity_cleaned AS
WITH staged AS (
    SELECT
        *,
        CASE
            WHEN product_normalized_value IS NOT NULL AND product_category IN ('mass', 'volume') THEN 'resolved'
            WHEN product_normalized_value IS NOT NULL AND product_category = 'energy' THEN 'partial'
            ELSE NULL
        END AS product_status,
        CASE
            WHEN product_category IN ('mass', 'volume', 'energy')
                 AND text_normalized_value IS NOT NULL
                 AND text_normalized_unit IS NOT NULL
            THEN product_base_unit = text_normalized_unit
            ELSE FALSE
        END AS comparable_unit_match,
        CASE
            WHEN product_category IN ('mass', 'volume', 'energy')
                 AND text_normalized_value IS NOT NULL
                 AND text_normalized_unit IS NOT NULL
                 AND product_base_unit = text_normalized_unit
            THEN abs(product_normalized_value - text_normalized_value)
                 <= CASE
                        WHEN text_subtype IN ('dual_unit_equivalent', 'compound_imperial_equivalent')
                        THEN greatest(1.0, 0.045 * greatest(abs(product_normalized_value), abs(text_normalized_value)))
                        ELSE greatest(0.01, {VALUE_MATCH_TOLERANCE} * greatest(abs(product_normalized_value), abs(text_normalized_value)))
                    END
            ELSE FALSE
        END AS comparable_value_match,
        CASE
            WHEN product_category IN ('mass', 'volume')
                 AND text_category = 'multipack_measure'
                 AND text_inner_normalized_value IS NOT NULL
                 AND product_base_unit = text_normalized_unit
            THEN abs(product_normalized_value - text_inner_normalized_value)
                 <= greatest(0.01, {VALUE_MATCH_TOLERANCE} * greatest(abs(product_normalized_value), abs(text_inner_normalized_value)))
            ELSE FALSE
        END AS multipack_inner_value_match,
        CASE
            WHEN product_quantity_numeric IS NOT NULL
            THEN coalesce(
                abs(product_quantity_numeric - measure_1_numeric)
                <= greatest(0.01, {VALUE_MATCH_TOLERANCE} * greatest(abs(product_quantity_numeric), abs(measure_1_numeric))),
                FALSE
            ) OR coalesce(
                abs(product_quantity_numeric - measure_2_numeric)
                <= greatest(0.01, {VALUE_MATCH_TOLERANCE} * greatest(abs(product_quantity_numeric), abs(measure_2_numeric))),
                FALSE
            ) OR coalesce(
                abs(product_quantity_numeric - measure_3_numeric)
                <= greatest(0.01, {VALUE_MATCH_TOLERANCE} * greatest(abs(product_quantity_numeric), abs(measure_3_numeric))),
                FALSE
            ) OR coalesce(
                abs(product_quantity_numeric - measure_4_numeric)
                <= greatest(0.01, {VALUE_MATCH_TOLERANCE} * greatest(abs(product_quantity_numeric), abs(measure_4_numeric))),
                FALSE
            )
            ELSE FALSE
        END AS raw_numeric_matches_any_text_measure,
        CASE
            WHEN product_quantity_numeric IS NOT NULL
                 AND text_category = 'multipack_measure'
                 AND text_normalized_value IS NOT NULL
            THEN (
                abs(product_quantity_numeric - text_normalized_value)
                <= greatest(0.01, {VALUE_MATCH_TOLERANCE} * greatest(abs(product_quantity_numeric), abs(text_normalized_value)))
            ) OR (
                text_inner_normalized_value IS NOT NULL
                AND abs(product_quantity_numeric - text_inner_normalized_value)
                    <= greatest(0.01, {VALUE_MATCH_TOLERANCE} * greatest(abs(product_quantity_numeric), abs(text_inner_normalized_value)))
            )
            WHEN product_quantity_numeric IS NOT NULL
                 AND text_category = 'count'
                 AND text_normalized_value IS NOT NULL
            THEN product_quantity_numeric = text_normalized_value
            WHEN product_quantity_numeric IS NOT NULL
                 AND text_normalized_value IS NOT NULL
            THEN abs(product_quantity_numeric - text_normalized_value)
                 <= greatest(0.01, {VALUE_MATCH_TOLERANCE} * greatest(abs(product_quantity_numeric), abs(text_normalized_value)))
            ELSE FALSE
        END AS raw_numeric_matches_text
    FROM quantity_cleaning_features
)
SELECT
    row_id,
    code,
    product_quantity_unit,
    product_quantity,
    quantity,
    CASE
        WHEN text_status = 'conflict' THEN 'conflict'
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('mass', 'volume', 'multipack_measure', 'energy')
             AND NOT comparable_value_match
             AND NOT multipack_inner_value_match
        THEN 'conflict'
        WHEN product_status = 'partial'
             AND text_status = 'partial'
             AND text_category = 'energy'
             AND comparable_value_match
        THEN 'partial'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND (multipack_inner_value_match OR comparable_value_match)
        THEN 'resolved'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category IN ('mass', 'volume')
             AND comparable_value_match
        THEN 'resolved'
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('count', 'packaging_only')
        THEN 'resolved'
        WHEN product_status = 'resolved' THEN 'resolved'
        WHEN product_quantity_numeric = 0
             AND text_status IN ('resolved', 'partial')
             AND (text_normalized_value IS NOT NULL OR text_category = 'packaging_only')
        THEN text_status
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND (raw_numeric_matches_text OR raw_numeric_matches_any_text_measure)
        THEN text_status
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND NOT (raw_numeric_matches_text OR raw_numeric_matches_any_text_measure)
        THEN 'conflict'
        WHEN text_status IS NOT NULL THEN text_status
        WHEN product_quantity_numeric IS NOT NULL AND product_base_unit IS NULL THEN 'unresolved'
        ELSE 'unresolved'
    END AS quantity_status,
    CASE
        WHEN text_status = 'conflict' THEN coalesce(text_category, product_category)
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND (multipack_inner_value_match OR comparable_value_match)
        THEN 'multipack_measure'
        WHEN product_status = 'resolved' THEN product_category
        WHEN product_status = 'partial'
             AND text_status = 'partial'
             AND text_category = 'energy'
             AND comparable_value_match
        THEN 'energy'
        WHEN text_status IS NOT NULL THEN text_category
        ELSE product_category
    END AS quantity_category,
    CASE
        WHEN text_status = 'conflict' THEN NULL
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND (multipack_inner_value_match OR comparable_value_match)
        THEN text_normalized_value
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('count', 'packaging_only')
        THEN product_normalized_value
        WHEN product_status = 'resolved' THEN product_normalized_value
        WHEN product_status = 'partial'
             AND text_status = 'partial'
             AND text_category = 'energy'
             AND comparable_value_match
        THEN product_normalized_value
        WHEN product_quantity_numeric = 0
             AND text_status = 'resolved'
             AND text_category = 'count'
        THEN text_normalized_value
        WHEN product_quantity_numeric = 0
             AND text_status = 'partial'
             AND text_category = 'packaging_only'
        THEN NULL
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND (raw_numeric_matches_text OR raw_numeric_matches_any_text_measure)
             AND text_category <> 'packaging_only'
        THEN text_normalized_value
        WHEN text_status = 'resolved'
             AND text_category <> 'packaging_only'
        THEN text_normalized_value
        WHEN text_status = 'partial'
             AND text_category = 'energy'
        THEN text_normalized_value
        ELSE NULL
    END AS normalized_value,
    CASE
        WHEN text_status = 'conflict' THEN NULL
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND (multipack_inner_value_match OR comparable_value_match)
        THEN text_normalized_unit
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('count', 'packaging_only')
        THEN product_base_unit
        WHEN product_status = 'resolved' THEN product_base_unit
        WHEN product_status = 'partial'
             AND text_status = 'partial'
             AND text_category = 'energy'
             AND comparable_value_match
        THEN product_base_unit
        WHEN product_quantity_numeric = 0
             AND text_status = 'resolved'
             AND text_category = 'count'
        THEN text_normalized_unit
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND (raw_numeric_matches_text OR raw_numeric_matches_any_text_measure)
             AND text_category <> 'packaging_only'
        THEN text_normalized_unit
        WHEN text_status = 'resolved'
             AND text_category <> 'packaging_only'
        THEN text_normalized_unit
        WHEN text_status = 'partial'
             AND text_category = 'energy'
        THEN text_normalized_unit
        ELSE NULL
    END AS normalized_unit,
    CASE
        WHEN text_status = 'conflict' THEN NULL
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category = 'count'
        THEN greatest(coalesce(text_normalized_value, 1.0), 1.0)
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category = 'packaging_only'
        THEN coalesce(text_pack_count, 1.0)
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
        THEN coalesce(text_pack_count, 1.0)
        WHEN product_status = 'resolved' THEN 1.0
        WHEN text_status = 'resolved' AND text_category = 'count' THEN 1.0
        WHEN text_status = 'partial' AND text_category = 'packaging_only' THEN coalesce(text_pack_count, 1.0)
        WHEN text_status = 'resolved' AND text_category = 'multipack_measure' THEN coalesce(text_pack_count, 1.0)
        WHEN text_status IN ('resolved', 'partial') THEN 1.0
        ELSE NULL
    END AS pack_count,
    CASE
        WHEN text_status = 'conflict' THEN NULL
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_item_descriptor IS NOT NULL
        THEN text_item_descriptor
        WHEN text_status IN ('resolved', 'partial') THEN text_item_descriptor
        ELSE NULL
    END AS item_descriptor,
    CASE
        WHEN text_status = 'conflict' THEN TRUE
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('mass', 'volume', 'multipack_measure', 'energy')
             AND NOT comparable_value_match
             AND NOT multipack_inner_value_match
        THEN TRUE
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND NOT (raw_numeric_matches_text OR raw_numeric_matches_any_text_measure)
        THEN TRUE
        ELSE FALSE
    END AS quantity_conflict_flag,
    CASE
        WHEN text_status = 'conflict' THEN text_note
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND text_subtype = 'total_plus_inner_multipack'
             AND comparable_value_match
        THEN 'structured total matches quantity total-plus-inner multipack'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND text_subtype = 'total_plus_count'
             AND comparable_value_match
        THEN 'structured total matches quantity total-plus-count'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND multipack_inner_value_match
        THEN 'product fields match inner quantity; total derived from quantity multipack'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'multipack_measure'
             AND comparable_value_match
        THEN 'structured total matches quantity multipack'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_subtype IN ('dual_unit_equivalent', 'compound_imperial_equivalent')
             AND text_category IN ('mass', 'volume')
             AND comparable_value_match
        THEN 'product fields agree with equivalent mixed-unit text'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category IN ('mass', 'volume')
             AND comparable_value_match
        THEN 'product fields and quantity agree'
        WHEN product_status = 'partial'
             AND text_status = 'partial'
             AND text_category = 'energy'
             AND comparable_value_match
        THEN 'energy captured consistently but not used for consolidation'
        WHEN product_status = 'resolved'
             AND text_status = 'resolved'
             AND text_category = 'count'
        THEN 'structured product fields used with count metadata from quantity'
        WHEN product_status = 'resolved'
             AND text_status = 'partial'
             AND text_category = 'packaging_only'
        THEN 'structured product fields used with packaging metadata from quantity'
        WHEN product_status = 'resolved'
             AND text_status IN ('resolved', 'partial')
             AND text_category IN ('mass', 'volume', 'multipack_measure', 'energy')
             AND NOT comparable_value_match
             AND NOT multipack_inner_value_match
        THEN 'conflict between product fields and quantity text'
        WHEN product_status = 'resolved'
        THEN 'structured product fields used'
        WHEN product_quantity_numeric = 0
             AND text_status = 'partial'
             AND text_category = 'packaging_only'
        THEN 'structured zero ignored; packaging text kept as partial metadata'
        WHEN product_quantity_numeric = 0
             AND text_status IN ('resolved', 'partial')
             AND text_normalized_value IS NOT NULL
        THEN 'structured zero ignored; quantity text used'
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND (raw_numeric_matches_text OR raw_numeric_matches_any_text_measure)
        THEN 'filled missing structure from quantity'
        WHEN product_quantity_numeric IS NOT NULL
             AND product_base_unit IS NULL
             AND text_status IN ('resolved', 'partial')
             AND NOT (raw_numeric_matches_text OR raw_numeric_matches_any_text_measure)
        THEN 'structured numeric value disagrees with quantity text'
        WHEN text_status IS NOT NULL THEN text_note
        WHEN product_quantity_numeric IS NOT NULL AND product_base_unit IS NULL THEN 'structured numeric value present, but no usable unit found'
        ELSE 'no usable quantity signal found'
    END AS quantity_note
FROM staged;
'''

con.execute(quantity_cleaned_sql)

review_sql = '''
CREATE OR REPLACE TEMP VIEW quantity_cleaning_debug AS
SELECT
    q.row_id,
    q.code,
    q.product_quantity_unit,
    q.product_quantity,
    q.quantity,
    q.quantity_status,
    q.quantity_category,
    q.normalized_value,
    q.normalized_unit,
    q.pack_count,
    q.item_descriptor,
    q.quantity_conflict_flag,
    q.quantity_note,
    f.quantity_normalized_raw AS raw_cleaned_quantity_text,
    f.quantity_normalized AS cleaned_quantity_text,
    f.quantity_ocr_cleanup_applied AS used_ocr_cleanup,
    f.quantity_has_ocr_like_numeric_token AS has_ocr_like_token,
    f.quantity_has_non_ascii_alpha AS has_non_ascii_token,
    f.quantity_uses_new_multilingual_alias AS used_multilingual_alias,
    f.mixed_resolution_system AS equivalence_system_used,
    f.text_subtype AS parse_path,
    b.quantity_status AS reference_status,
    b.quantity_category AS reference_category,
    b.quantity_note AS reference_note
FROM quantity_cleaned q
JOIN quantity_cleaning_features f USING (row_id)
LEFT JOIN quantity_cleaned_reference b USING (row_id);
'''

con.execute(review_sql)

show_query("Final cleaned schema", "DESCRIBE quantity_cleaned")
show_query(
    "Final status counts",
    '''
    SELECT quantity_status, COUNT(*) AS rows
    FROM quantity_cleaned
    GROUP BY 1
    ORDER BY rows DESC, quantity_status
    '''
)
show_query(
    "Final category counts",
    '''
    SELECT coalesce(quantity_category, 'null') AS quantity_category, COUNT(*) AS rows
    FROM quantity_cleaned
    GROUP BY 1
    ORDER BY rows DESC, quantity_category
    '''
)


In [ ]:
# This cell shows real row examples from the final cleaned output view.
# It keeps the focus on the columns a downstream user actually needs.

show_query(
    "Sample cleaned rows by status",
    '''
    WITH ranked AS (
        SELECT
            quantity_status,
            code,
            product_quantity_unit,
            product_quantity,
            quantity,
            quantity_category,
            normalized_value,
            normalized_unit,
            pack_count,
            item_descriptor,
            quantity_conflict_flag,
            quantity_note,
            row_number() OVER (PARTITION BY quantity_status ORDER BY row_id) AS rn
        FROM quantity_cleaned
    )
    SELECT
        quantity_status,
        code,
        product_quantity_unit,
        product_quantity,
        quantity,
        quantity_category,
        normalized_value,
        normalized_unit,
        pack_count,
        item_descriptor,
        quantity_conflict_flag,
        quantity_note
    FROM ranked
    WHERE rn <= 8
    ORDER BY quantity_status, rn
    '''
)

show_query(
    "Sample rows where quantity text helped the final result",
    '''
    SELECT
        code,
        product_quantity_unit,
        product_quantity,
        quantity,
        quantity_status,
        quantity_category,
        normalized_value,
        normalized_unit,
        pack_count,
        item_descriptor,
        quantity_note
    FROM quantity_cleaning_debug
    WHERE quantity_note IN (
        'filled missing structure from quantity',
        'structured zero ignored; quantity text used',
        'derived total from quantity multipack',
        'count descriptor from quantity',
        'descriptor plus measure from quantity',
        'descriptor plus measure from quantity after OCR cleanup',
        'equivalent dual-unit text normalized',
        'equivalent dual-unit text normalized after OCR cleanup',
        'total size plus count metadata from quantity'
    )
    ORDER BY row_id
    LIMIT 25
    '''
)


In [ ]:

# This cell gives a compact QA summary for the most important rule families in the current cleaner.

show_query(
    "QA summary",
    '''
    SELECT 'rows_with_conflicts' AS metric, COUNT(*) AS value
    FROM quantity_cleaned
    WHERE quantity_status = 'conflict'
    UNION ALL
    SELECT 'rows_resolved_from_dual_unit_equivalence', COUNT(*)
    FROM quantity_cleaning_debug
    WHERE parse_path = 'dual_unit_equivalent' AND quantity_status = 'resolved'
    UNION ALL
    SELECT 'rows_resolved_from_embedded_plain_multipack', COUNT(*)
    FROM quantity_cleaning_debug
    WHERE parse_path = 'embedded_plain_multipack' AND quantity_status = 'resolved'
    UNION ALL
    SELECT 'rows_resolved_from_embedded_descriptor_multipack', COUNT(*)
    FROM quantity_cleaning_debug
    WHERE parse_path = 'embedded_descriptor_multipack' AND quantity_status = 'resolved'
    UNION ALL
    SELECT 'rows_resolved_from_compound_imperial_equivalence', COUNT(*)
    FROM quantity_cleaning_debug
    WHERE parse_path = 'compound_imperial_equivalent' AND quantity_status = 'resolved'
    UNION ALL
    SELECT 'rows_resolved_from_total_plus_count', COUNT(*)
    FROM quantity_cleaning_debug
    WHERE parse_path = 'total_plus_count' AND quantity_status = 'resolved'
    UNION ALL
    SELECT 'rows_resolved_from_total_plus_inner_multipack', COUNT(*)
    FROM quantity_cleaning_debug
    WHERE parse_path = 'total_plus_inner_multipack' AND quantity_status = 'resolved'
    UNION ALL
    SELECT 'rows_resolved_after_safe_ocr_cleanup', COUNT(*)
    FROM quantity_cleaning_debug
    WHERE used_ocr_cleanup AND quantity_status = 'resolved'
    UNION ALL
    SELECT 'rows_using_approved_multilingual_aliases', COUNT(*)
    FROM quantity_cleaning_debug
    WHERE used_multilingual_alias
    UNION ALL
    SELECT 'rows_resolved_with_imperial_fl_oz_match', COUNT(*)
    FROM quantity_cleaning_debug
    WHERE equivalence_system_used = 'imperial' AND quantity_status = 'resolved'
    UNION ALL
    SELECT 'rows_left_unresolved_as_household_units', COUNT(*)
    FROM quantity_cleaning_debug
    WHERE quantity_category = 'household_unit'
    UNION ALL
    SELECT 'rows_left_unresolved_as_contextual_mixed', COUNT(*)
    FROM quantity_cleaning_debug
    WHERE parse_path = 'contextual_mixed_unresolved'
    UNION ALL
    SELECT 'rows_with_unsupported_foreign_unit_reason', COUNT(*)
    FROM quantity_cleaning_debug
    WHERE parse_path = 'unsupported_foreign_unit'
    UNION ALL
    SELECT 'rows_split_into_mixed_conflict', COUNT(*)
    FROM quantity_cleaning_debug
    WHERE parse_path = 'mixed_conflict'
    UNION ALL
    SELECT 'rows_split_into_mixed_unresolved', COUNT(*)
    FROM quantity_cleaning_debug
    WHERE parse_path = 'mixed_unresolved'
    '''
)


## Final Table EDA

The earlier cells built the final cleaned view and the optional debug helpers. This section explores what the current cleaner is actually doing in practice:

- how many rows are `resolved`, `partial`, `unresolved`, or `conflict`
- how the final categories are distributed
- which `quantity_note` messages and parse paths are actually being used
- how often the cleaned numeric/unit fields are populated
- what unresolved and conflict rows still look like after cleaning


In [ ]:
# This cell gives the high-level shape of the final cleaned table.
# It shows:
# - output column coverage
# - final status counts
# - final category counts
# - how status and category combine together

show_query(
    "Final table output coverage",
    '''
    SELECT 'quantity_status' AS field,
           COUNT(*) FILTER (WHERE quantity_status IS NOT NULL) AS non_null_count,
           COUNT(DISTINCT quantity_status) FILTER (WHERE quantity_status IS NOT NULL) AS distinct_non_null_count
    FROM quantity_cleaned
    UNION ALL
    SELECT 'quantity_category',
           COUNT(*) FILTER (WHERE quantity_category IS NOT NULL),
           COUNT(DISTINCT quantity_category) FILTER (WHERE quantity_category IS NOT NULL)
    FROM quantity_cleaned
    UNION ALL
    SELECT 'normalized_value',
           COUNT(*) FILTER (WHERE normalized_value IS NOT NULL),
           COUNT(DISTINCT normalized_value) FILTER (WHERE normalized_value IS NOT NULL)
    FROM quantity_cleaned
    UNION ALL
    SELECT 'normalized_unit',
           COUNT(*) FILTER (WHERE normalized_unit IS NOT NULL),
           COUNT(DISTINCT normalized_unit) FILTER (WHERE normalized_unit IS NOT NULL)
    FROM quantity_cleaned
    UNION ALL
    SELECT 'pack_count',
           COUNT(*) FILTER (WHERE pack_count IS NOT NULL),
           COUNT(DISTINCT pack_count) FILTER (WHERE pack_count IS NOT NULL)
    FROM quantity_cleaned
    UNION ALL
    SELECT 'item_descriptor',
           COUNT(*) FILTER (WHERE item_descriptor IS NOT NULL),
           COUNT(DISTINCT item_descriptor) FILTER (WHERE item_descriptor IS NOT NULL)
    FROM quantity_cleaned
    UNION ALL
    SELECT 'quantity_conflict_flag',
           COUNT(*) FILTER (WHERE quantity_conflict_flag IS NOT NULL),
           COUNT(DISTINCT quantity_conflict_flag) FILTER (WHERE quantity_conflict_flag IS NOT NULL)
    FROM quantity_cleaned
    UNION ALL
    SELECT 'quantity_note',
           COUNT(*) FILTER (WHERE quantity_note IS NOT NULL),
           COUNT(DISTINCT quantity_note) FILTER (WHERE quantity_note IS NOT NULL)
    FROM quantity_cleaned
    '''
)

show_query(
    "Final status counts",
    '''
    SELECT quantity_status, COUNT(*) AS rows
    FROM quantity_cleaned
    GROUP BY 1
    ORDER BY rows DESC, quantity_status
    '''
)

show_query(
    "Final category counts",
    '''
    SELECT COALESCE(quantity_category, 'null') AS quantity_category, COUNT(*) AS rows
    FROM quantity_cleaned
    GROUP BY 1
    ORDER BY rows DESC, quantity_category
    '''
)

show_query(
    "Status by category",
    '''
    SELECT quantity_status,
           COALESCE(quantity_category, 'null') AS quantity_category,
           COUNT(*) AS rows
    FROM quantity_cleaned
    GROUP BY 1, 2
    ORDER BY quantity_status, rows DESC, quantity_category
    '''
)


In [ ]:
# This cell focuses on the explanation columns because they show how the cleaner reached its result.
# It answers:
# - how many distinct final messages we actually ended up using
# - which messages are most common
# - which internal parse paths are most common in the debug view

show_query(
    "quantity_note distinct-count summary",
    '''
    SELECT COUNT(DISTINCT quantity_note) AS distinct_quantity_note_messages
    FROM quantity_cleaned
    WHERE quantity_note IS NOT NULL
    '''
)

show_query(
    "quantity_note counts",
    '''
    SELECT quantity_note, COUNT(*) AS rows
    FROM quantity_cleaned
    GROUP BY 1
    ORDER BY rows DESC, quantity_note
    '''
)

show_query(
    "quantity_note by final status and category",
    '''
    SELECT quantity_note,
           quantity_status,
           COALESCE(quantity_category, 'null') AS quantity_category,
           COUNT(*) AS rows
    FROM quantity_cleaned
    GROUP BY 1, 2, 3
    ORDER BY quantity_note, rows DESC, quantity_status, quantity_category
    '''
)

show_query(
    "parse_path counts",
    '''
    SELECT parse_path, COUNT(*) AS rows
    FROM quantity_cleaning_debug
    GROUP BY 1
    ORDER BY rows DESC, parse_path
    '''
)


In [ ]:
# This cell explores the structured outputs that downstream users would actually consume.
# It shows:
# - which normalized units are most common
# - which descriptors are being used
# - how pack_count behaves across categories

show_query(
    "normalized_unit counts",
    '''
    SELECT COALESCE(normalized_unit, 'null') AS normalized_unit, COUNT(*) AS rows
    FROM quantity_cleaned
    GROUP BY 1
    ORDER BY rows DESC, normalized_unit
    '''
)

show_query(
    "item_descriptor counts",
    '''
    SELECT COALESCE(item_descriptor, 'null') AS item_descriptor, COUNT(*) AS rows
    FROM quantity_cleaned
    GROUP BY 1
    ORDER BY rows DESC, item_descriptor
    '''
)

show_query(
    "pack_count profile by category",
    '''
    SELECT COALESCE(quantity_category, 'null') AS quantity_category,
           COUNT(*) FILTER (WHERE pack_count IS NOT NULL) AS rows_with_pack_count,
           COUNT(*) FILTER (WHERE pack_count = 1) AS rows_with_pack_count_1,
           COUNT(*) FILTER (WHERE pack_count > 1) AS rows_with_pack_count_gt_1,
           COUNT(DISTINCT pack_count) FILTER (WHERE pack_count IS NOT NULL) AS distinct_pack_count_values
    FROM quantity_cleaned
    GROUP BY 1
    ORDER BY rows_with_pack_count DESC, quantity_category
    '''
)

show_query(
    "Most common pack_count values",
    '''
    SELECT pack_count, COUNT(*) AS rows
    FROM quantity_cleaned
    WHERE pack_count IS NOT NULL
    GROUP BY 1
    ORDER BY rows DESC, pack_count
    '''
)


In [ ]:

# This cell drills into the rows that still need attention after cleaning.
# It uses the smaller debug view so we can inspect the cleaned text and parse path without overwhelming the table.

show_query(
    "Unresolved and conflict note counts",
    '''
    SELECT quantity_status,
           quantity_note,
           COUNT(*) AS rows
    FROM quantity_cleaned
    WHERE quantity_status IN ('unresolved', 'conflict')
    GROUP BY 1, 2
    ORDER BY quantity_status, rows DESC, quantity_note
    '''
)

show_query(
    "Sample conflict rows",
    '''
    SELECT
        code,
        product_quantity_unit,
        product_quantity,
        quantity,
        cleaned_quantity_text,
        equivalence_system_used,
        parse_path,
        quantity_category,
        quantity_conflict_flag,
        quantity_note
    FROM quantity_cleaning_debug
    WHERE quantity_status = 'conflict'
    ORDER BY row_id
    '''
)

show_query(
    "Sample partial rows",
    '''
    SELECT
        code,
        product_quantity_unit,
        product_quantity,
        quantity,
        cleaned_quantity_text,
        parse_path,
        quantity_category,
        normalized_value,
        normalized_unit,
        pack_count,
        item_descriptor,
        quantity_note
    FROM quantity_cleaning_debug
    WHERE quantity_status = 'partial'
    ORDER BY row_id
    LIMIT 15
    '''
)


In [ ]:
# This cell performs the dataset-driven alias-mining pass from the rows that are still unresolved.
# It does not auto-approve anything. It only surfaces frequent unknown tokens for manual review.

show_query(
    "Alias mining candidates from unresolved numeric-like rows",
    f'''
    WITH candidate_rows AS (
        SELECT row_id, cleaned_quantity_text
        FROM quantity_cleaning_debug
        WHERE quantity_status = 'unresolved'
          AND cleaned_quantity_text IS NOT NULL
          AND (regexp_matches(cleaned_quantity_text, '\\d') OR parse_path IN ('mixed_unresolved', 'unsupported_foreign_unit', 'ocr_like_numeric_token'))
    ),
    stripped AS (
        SELECT
            row_id,
            regexp_replace(cleaned_quantity_text, '{MEASURE_ANY_REGEX}', ' ', 'g') AS no_measures
        FROM candidate_rows
    ),
    tokenized AS (
        SELECT
            row_id,
            unnest(
                string_split(
                    trim(
                        regexp_replace(
                            regexp_replace(
                                regexp_replace(no_measures, '{NUMBER_PATTERN}', ' ', 'g'),
                                '[^a-z]+',
                                ' ',
                                'g'
                            ),
                            '\\s+',
                            ' ',
                            'g'
                        )
                    ),
                    ' '
                )
            ) AS token
        FROM stripped
    )
    SELECT token, COUNT(*) AS rows
    FROM tokenized
    WHERE token <> ''
      AND token NOT IN (SELECT token FROM measure_aliases)
      AND token NOT IN (SELECT token FROM descriptor_aliases)
      AND token NOT IN (SELECT token FROM household_aliases)
      AND token NOT IN ({ALIAS_MINING_STOPWORDS_SQL_LIST})
    GROUP BY 1
    ORDER BY rows DESC, token
    '''
)

show_query(
    "Sample unresolved rows for top alias candidates",
    f'''
    WITH candidate_rows AS (
        SELECT row_id, cleaned_quantity_text
        FROM quantity_cleaning_debug
        WHERE quantity_status = 'unresolved'
          AND cleaned_quantity_text IS NOT NULL
          AND (regexp_matches(cleaned_quantity_text, '\\d') OR parse_path IN ('mixed_unresolved', 'unsupported_foreign_unit', 'ocr_like_numeric_token'))
    ),
    stripped AS (
        SELECT
            row_id,
            regexp_replace(cleaned_quantity_text, '{MEASURE_ANY_REGEX}', ' ', 'g') AS no_measures
        FROM candidate_rows
    ),
    tokenized AS (
        SELECT
            row_id,
            unnest(
                string_split(
                    trim(
                        regexp_replace(
                            regexp_replace(
                                regexp_replace(no_measures, '{NUMBER_PATTERN}', ' ', 'g'),
                                '[^a-z]+',
                                ' ',
                                'g'
                            ),
                            '\\s+',
                            ' ',
                            'g'
                        )
                    ),
                    ' '
                )
            ) AS token
        FROM stripped
    ),
    top_tokens AS (
        SELECT token
        FROM tokenized
        WHERE token <> ''
          AND token NOT IN (SELECT token FROM measure_aliases)
          AND token NOT IN (SELECT token FROM descriptor_aliases)
          AND token NOT IN (SELECT token FROM household_aliases)
          AND token NOT IN ({ALIAS_MINING_STOPWORDS_SQL_LIST})
        GROUP BY 1
        ORDER BY COUNT(*) DESC, token
        LIMIT 10
    ),
    ranked AS (
        SELECT
            t.token,
            r.code,
            r.quantity,
            r.cleaned_quantity_text,
            r.quantity_note,
            row_number() OVER (PARTITION BY t.token ORDER BY r.row_id) AS rn
        FROM top_tokens t
        JOIN quantity_cleaning_debug r
          ON contains(' ' || r.cleaned_quantity_text || ' ', ' ' || t.token || ' ')
        WHERE r.quantity_status = 'unresolved'
    )
    SELECT token, code, quantity, cleaned_quantity_text, quantity_note
    FROM ranked
    WHERE rn <= 5
    ORDER BY token, rn
    '''
)


In [ ]:

# This cell audits how the current cleaner differs from the internal reference snapshot.
# It is optional review material, not a second final output table.

con.execute(
    '''
    CREATE OR REPLACE TEMP VIEW quantity_change_audit AS
    WITH joined AS (
        SELECT
            row_id,
            code,
            product_quantity_unit,
            product_quantity,
            quantity,
            raw_cleaned_quantity_text,
            cleaned_quantity_text,
            reference_status,
            reference_category,
            reference_note,
            quantity_status,
            quantity_category,
            quantity_note,
            normalized_value,
            normalized_unit,
            pack_count,
            item_descriptor,
            parse_path,
            used_ocr_cleanup,
            used_multilingual_alias,
            equivalence_system_used
        FROM quantity_cleaning_debug
    )
    SELECT 'dual_unit_equivalence' AS rule_name, * FROM joined WHERE parse_path = 'dual_unit_equivalent'
    UNION ALL
    SELECT 'embedded_plain_multipack', * FROM joined WHERE parse_path = 'embedded_plain_multipack'
    UNION ALL
    SELECT 'embedded_descriptor_multipack', * FROM joined WHERE parse_path = 'embedded_descriptor_multipack'
    UNION ALL
    SELECT 'compound_imperial_equivalence', * FROM joined WHERE parse_path = 'compound_imperial_equivalent'
    UNION ALL
    SELECT 'total_plus_count', * FROM joined WHERE parse_path = 'total_plus_count'
    UNION ALL
    SELECT 'total_plus_inner_multipack', * FROM joined WHERE parse_path = 'total_plus_inner_multipack'
    UNION ALL
    SELECT 'descriptor_multipack', * FROM joined WHERE parse_path = 'descriptor_multipack'
    UNION ALL
    SELECT 'contextual_mixed_unresolved', * FROM joined WHERE parse_path = 'contextual_mixed_unresolved'
    UNION ALL
    SELECT 'mixed_measure_conflict_split', * FROM joined WHERE parse_path = 'mixed_conflict'
    UNION ALL
    SELECT 'mixed_measure_unresolved_split', * FROM joined WHERE parse_path = 'mixed_unresolved'
    UNION ALL
    SELECT 'safe_ocr_cleanup', * FROM joined WHERE used_ocr_cleanup
    UNION ALL
    SELECT 'approved_multilingual_alias', * FROM joined WHERE used_multilingual_alias
    UNION ALL
    SELECT 'explicit_household_reason', * FROM joined WHERE parse_path = 'household_unit';
    '''
)

show_query(
    "Change-audit summary",
    '''
    SELECT
        rule_name,
        COUNT(*) AS rows_in_rule,
        COUNT(*) FILTER (WHERE reference_status IS DISTINCT FROM quantity_status) AS changed_rows,
        COUNT(*) FILTER (WHERE reference_status = 'unresolved' AND quantity_status = 'resolved') AS unresolved_to_resolved,
        COUNT(*) FILTER (WHERE reference_status = 'conflict' AND quantity_status = 'resolved') AS conflict_to_resolved,
        COUNT(*) FILTER (WHERE reference_status = 'unresolved' AND quantity_status = 'conflict') AS unresolved_to_conflict
    FROM quantity_change_audit
    GROUP BY 1
    ORDER BY rows_in_rule DESC, rule_name
    '''
)

show_query(
    "Change-audit samples",
    '''
    WITH ranked AS (
        SELECT
            rule_name,
            reference_status,
            quantity_status,
            parse_path,
            reference_note,
            quantity_note,
            code,
            product_quantity_unit,
            product_quantity,
            quantity,
            cleaned_quantity_text,
            quantity_category,
            normalized_value,
            normalized_unit,
            pack_count,
            item_descriptor,
            equivalence_system_used,
            row_number() OVER (PARTITION BY rule_name ORDER BY row_id) AS rn
        FROM quantity_change_audit
    )
    SELECT
        rule_name,
        reference_status,
        quantity_status,
        parse_path,
        reference_note,
        quantity_note,
        code,
        product_quantity_unit,
        product_quantity,
        quantity,
        cleaned_quantity_text,
        equivalence_system_used,
        quantity_category,
        normalized_value,
        normalized_unit,
        pack_count,
        item_descriptor
    FROM ranked
    WHERE rn <= 6
    ORDER BY rule_name, rn
    '''
)
